In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q trl # datasets # bitsandbytes
!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 10.7 MB/s eta 0:00:00a 0:00:01


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
!hf auth login --token $HF_TOKEN

A new version of huggingface_hub (1.7.1) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `sd` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `sd`


In [ ]:
import os
import json
import random
import torch

from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

# ============================================================
# Login
# ============================================================

#login()  # or login(token=os.environ["HF_TOKEN"])

# ============================================================
# Config
# ============================================================

MODEL_ID = "Lyte/tiny-aya-darija-v3"
DATASET_ID = "Lyte/moroccan-instruct-59k"

SEED = 42
VAL_RATIO = 0.05
BIDIRECTIONAL_FRACTION = 0.20

random.seed(SEED)

In [ ]:
SYS_EN_DR = (
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the Darija translation."
)

SYS_DR_EN = (
    "Translate the following Moroccan Darija (الدارجة المغربية) text into English. "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the English translation."
)

# ============================================================
# Manual fix examples
# ============================================================

MANUAL_FIXES = [
    # --- exact failures from your examples ---
    {
        "source": "You are a person who loves ice cream and you adore dogs. How excited are you that there is a new ice cream truck in town that caters to dogs.",
        "target": "نتا واحد الشخص كيبغي الآيس كريم وكيعشق الكلاب. شحال فرحان حيث كاين كاميو جديد ديال الآيس كريم فالمدينة وكيدير حتى حاجات للكلاب؟",
    },
    {
        "source": "I am absolutely over the moon! A new ice cream truck in town is always a reason to celebrate, but one that caters to dogs? That's just the icing on the cake - or should I say, the sprinkles on the ice cream cone! I have a furry best friend at home, a sweet little pup named Max, and I know he's going to go wild for this. I can already imagine the look on his face when he tries his first dog-friendly ice cream. I'm planning on being the first in line when the truck opens, with Max by my side, of course. I've heard they have flavors like peanut butter and banana, and even a special \"pup-cup\" just for our canine friends. This is going to be the best summer ever!",
        "target": "أنا فرحان بزاف! كاميو جديد ديال الآيس كريم فالمدينة بوحدو سبب باش نفرحو، ولكن واحد كيدير حتى للكلاب؟ هادي هيا قمة الفرحة! عندي فالدّار صاحب عزيز عليا، جرو صغير وحلو سميتو ماكس، وعارف بلي غادي يفرح بزاف بهادشي. نقدر نتخايل من دابا الوجه ديالو ملي يذوق أول آيس كريم مناسب للكلاب. ناوي نكون أول واحد فالصف ملي يحل الكاميو، وماكس حدايا بطبيعة الحال. سمعت بلي عندهم نكهات بحال زبدة الكاوكاو والموز، وحتى واحد \"pup-cup\" خاص غير لصحابنا الكلاب. هاد الصيف غادي يكون أحسن صيف داز عليا!",
    },
    {
        "source": "Who is Max?",
        "target": "شكون هوا ماكس؟",
    },
    {
        "source": "Max is my lovable and adorable golden retriever. He's about three years old, with the fluffiest coat and the biggest brown eyes you've ever seen. He's such a sweetheart, always eager to please and playful to a fault. Max loves chasing after balls, snuggling on the couch, and getting belly rubs. And, of course, he's a huge fan of treats - which is why I know he's going to go crazy for the ice cream truck! Max is more than just a pet, he's my constant companion and best friend. We do everything together, from going on walks to playing fetch in the park. He's such a big part of my life, and I couldn't imagine it without him.",
        "target": "ماكس هوا الكلب ديالي من نوع golden retriever، زوين ومحبوب بزاف. عندو تقريبا تلت سنين، وعندو زغب منفوش وعينين كبار وبنيين. قلبو بيض وديما باغي يفرحك وكيبغي يلعب بزاف. ماكس كيبغي يجري مور الكواري، ويتكمش فالفوتاي، وكيعجبو تحك ليه كرشو. وبالطبع كيحب السقاطة، وداكشي علاش عارف بلي غادي يفرح بزاف بكاميو الآيس كريم. ماكس ماشي غير حيوان أليف، هوا صاحبي الدائم وأحسن رفيق عندي. كنديرو كلشي بجوج، من التمشية حتى للعب فالبارك. عندو بلاصة كبيرة فحياتي وما نقدرش نتصور حياتي بلا بيه.",
    },
    {
        "source": "Who do you think will be more excited when you go to the ice cream truck, you or Max?",
        "target": "شكون فبالك غادي يكون فرحان كثر ملي تمشيو لكاميو ديال الآيس كريم، نتا ولا ماكس؟",
    },
    {
        "source": "I think it's going to be a close call, but I'm pretty sure Max is going to be even more excited than I am! As soon as he catches a whiff of those delicious ice cream smells wafting from the truck, he's going to start bouncing up and down and pulling on his leash, trying to get closer to the source of the treats. And when he sees the colorful truck and hears the music playing, he's going to lose his mind! He'll probably start barking and spinning around in circles, his tail wagging so hard it might fall off. I'll be excited too, of course, but I think Max's enthusiasm is going to be infectious - he'll probably get me even more pumped up than I already am!",
        "target": "كنظن غادي تكون متقاربة بيناتنا، ولكن متأكد تقريبا بلي ماكس غادي يكون فرحان كثر مني! غير يشم ديك الريحة الزوينة ديال الآيس كريم اللي خارجة من الكاميو، غادي يبدا كينقز ويشد فالسلسلة ديالو باش يقرب لبلاصة الماكلة. وملي يشوف الكاميو الملوّن ويسمع الموسيقى، غادي يتحمس بزاف! غالبا غادي يبدا ينبح ويدور فبلاصتو، وذنبو كيتحرك بقوة. أنا حتى أنا غادي نفرح، بطبيعة الحال، ولكن كنظن الحماس ديال ماكس غادي يتعدى ليا حتى أنا ويزيد يشعل فيا الحماسة كثر من دابا.",
    },

    # --- exact-string / placeholder preservation ---
    {
        "source": "Your response should contain less than 50 words. Do not include keywords [forbidden_words] in the response. [forbidden_words] are: forbidden, restricted, not allowed, prohibited, banned. Finish your response with this exact phrase [ender]. No other words should follow this phrase. [ender] is \"Is there anything else I can help with?\"",
        "target": "الجواب ديالك خاصو يكون أقل من 50 كلمة. ما تدخلش الكلمات المفتاحية [forbidden_words] فالجواب. [forbidden_words] هما: forbidden, restricted, not allowed, prohibited, banned. سالي الجواب بهاد العبارة بالضبط [ender]. ما خص حتى كلمة تجي من بعد هاد العبارة. [ender] هيا \"Is there anything else I can help with?\"",
    },
    {
        "source": "Sure I can help with that Is there anything else I can help with?",
        "target": "مرحبا نقدر نعاونك فهادشي Is there anything else I can help with?",
    },
    {
        "source": "Your response should contain exactly 5 bullet points. Use the markdown bullet points such as: * This is point 1. Additionally, your entire response should be in all lowercase letters. no capital letters are allowed.",
        "target": "الجواب ديالك خاص يكون فيه بالضبط 5 ديال النقط. استعمل markdown bullet points بحال هكا: * هادي النقطة 1. وزيد، الجواب كامل خاصو يكون غير بحروف صغار. ما تستعمل حتى حرف كبير.",
    },
    {
        "source": "* this is point one\n* this is point two\n* this is point three\n* this is point four\n* this is point five",
        "target": "* هادي هيا النقطة اللولة\n* هادي هيا النقطة الثانية\n* هادي هيا النقطة الثالثة\n* هادي هيا النقطة الرابعة\n* هادي هيا النقطة الخامسة",
    },
    {
        "source": "Return exactly \"OK\". Do not add any other words.",
        "target": "رجّع ليا بالضبط \"OK\". ما تزيد حتى كلمة خرى.",
    },
    {
        "source": "OK",
        "target": "OK",
    },
    {
        "source": "Finish your answer with \"DONE\". Do not add anything after it.",
        "target": "سالي الجواب ديالك ب\"DONE\". ما تزيد والو من بعدو.",
    },
    {
        "source": "DONE",
        "target": "DONE",
    },
    {
        "source": "Do not translate anything inside backticks: `pip install transformers`",
        "target": "ما تترجم حتى حاجة داخل backticks: `pip install transformers`",
    },
    {
        "source": "Keep the placeholder [topic] exactly as it is.",
        "target": "خلي placeholder [topic] بالضبط كيف ما هوا.",
    },
    {
        "source": "Keep the variable names user_name and account_id unchanged.",
        "target": "خلي أسامي المتغيرات user_name و account_id بلا تبدال.",
    },
    {
        "source": "Do not change the markdown headings.",
        "target": "ما تبدلش markdown headings.",
    },
    {
        "source": "The final line must be exactly: END",
        "target": "السطر الأخير خاصو يكون بالضبط: END",
    },
    {
        "source": "END",
        "target": "END",
    },

    # --- named entities / places / food / numbers ---
    {
        "source": "What is the capital of Peru?",
        "target": "شنو هيا عاصمة بيرو؟",
    },
    {
        "source": "The capital of Peru is Lima.",
        "target": "عاصمة بيرو هيا ليما.",
    },
    {
        "source": "How many calories are in a Big Mac?",
        "target": "شحال من كالوري كاينين فـ Big Mac؟",
    },
    {
        "source": "My order number is 48291.",
        "target": "رقم الطلب ديالي هوا 48291.",
    },
    {
        "source": "The meeting is on Tuesday at 3:30 PM.",
        "target": "الاجتماع غادي يكون نهار الثلاثاء مع 3:30 ديال لعشية.",
    },

    # --- semantic grounding / no hallucination ---
    {
        "source": "We need to dig into the details of how the explorers communicated with the Aztecs.",
        "target": "خاصنا ندخلو فالتفاصيل ديال كيفاش المستكشفين تواصلو مع الأزتيك.",
    },
    {
        "source": "I was thinking about our project on the Spanish conquest and language.",
        "target": "كنت كنفكر فالمشروع ديالنا على الغزو الإسباني واللغة.",
    },
    {
        "source": "Let's meet next week and brainstorm some ideas.",
        "target": "يالله نتلاقاو الأسبوع الجاي ونفكرو فشي أفكار.",
    },
    {
        "source": "This is going to be epic!",
        "target": "هادشي غادي يكون زوين بزاف!",
    },
    {
        "source": "He is my best friend and constant companion.",
        "target": "هوا أحسن صاحب عندي وديما معايا.",
    },
    {
        "source": "As soon as he smells the treats, he starts jumping around.",
        "target": "غير يشم الماكلة، كيبدا كينقز ويدور.",
    },
    {
        "source": "I can already imagine the look on his face.",
        "target": "نقدر نتخايل من دابا شحال غادية تبان الفرحة فوجهو.",
    },

    # --- instructions / structure / literal fidelity ---
    {
        "source": "Answer in one sentence only.",
        "target": "جاوب غير بجملة وحدة.",
    },
    {
        "source": "Do not explain your reasoning.",
        "target": "ما تشرحش طريقة التفكير ديالك.",
    },
    {
        "source": "Use exactly three sentences.",
        "target": "استعمل بالضبط 3 جمل.",
    },
    {
        "source": "Write the answer as a numbered list.",
        "target": "كتب الجواب على شكل ليستة مرقمة.",
    },
    {
        "source": "Do not use emojis.",
        "target": "ما تستعمل حتى إيموجي.",
    },
    {
        "source": "Preserve the quotation marks exactly.",
        "target": "خلي علامات التنصيص بالضبط كيف ما هما.",
    },

    # --- idioms & figurative language (round 2) ---
    {
        "source": "It's raining cats and dogs outside.",
        "target": "كتصب الشتا خيط من السما برا.",
    },
    {
        "source": "She let the cat out of the bag about the surprise party.",
        "target": "فرشات البلان ديال المفاجأة ديال الحفلة.",
    },
    {
        "source": "He hit the nail on the head with that comment.",
        "target": "جابها لاصقة بهاد التعليق.",
    },

    # --- named entities / brands / locations (round 2) ---
    {
        "source": "I bought a new iPhone 15 Pro Max from the Apple Store in Casablanca.",
        "target": "شريت iPhone 15 Pro Max جديد من Apple Store فكازا.",
    },
    {
        "source": "Lionel Messi scored a hat-trick against Real Madrid last night.",
        "target": "ليونيل ميسي دار هاتريك ضد ريال مدريد البارح فالليل.",
    },
    {
        "source": "We watched a Netflix documentary about the Amazon rainforest.",
        "target": "تفرجنا فوثائقي على Netflix على الغابة الاستوائية ديال الأمازون.",
    },
    {
        "source": "She works at Google as a software engineer in Mountain View, California.",
        "target": "كتخدم فـ Google كمهندسة برمجيات فـ Mountain View، كاليفورنيا.",
    },
    {
        "source": "The Boeing 747 landed safely at Mohammed V International Airport.",
        "target": "الطيارة ديال Boeing 747 هبطات بسلام فمطار محمد الخامس الدولي.",
    },

    # --- numbers / dates / units: do NOT change values or currency ---
    {
        "source": "The package weighs 3.5 kg and costs $49.99 plus shipping.",
        "target": "الكولية كتوزن 3.5 كيلو والثمن ديالها $49.99 زائد مصاريف التوصيل.",
    },
    {
        "source": "Her flight departs at 07:45 AM on March 22, 2025, from gate B12.",
        "target": "الطيارة ديالها غتطير مع 07:45 دالصباح نهار 22 مارس 2025 من البوابة B12.",
    },
    {
        "source": "The recipe calls for 2 cups of flour, 1/2 teaspoon of salt, and 350°F for 25 minutes.",
        "target": "الوصفة كتحتاج 2 كيسان ديال الفورص، نص معيلقة صغيرة ديال الملحة، و 350°F مدة 25 دقيقة.",
    },
    {
        "source": "The price is $199.99, not €199.99.",
        "target": "الثمن هوا $199.99، ماشي €199.99.",
    },
    {
        "source": "The temperature outside is 95°F.",
        "target": "الحرارة بالبرّا هيا 95°F.",
    },
    {
        "source": "The total comes to £75.50 including VAT.",
        "target": "المجموع كيوصل لـ £75.50 مع الضريبة.",
    },

    # --- code / technical: preserve EXACTLY, do not change values ---
    {
        "source": "Run `git clone https://github.com/user/repo.git` and then `cd repo && npm install`.",
        "target": "دير `git clone https://github.com/user/repo.git` ومن بعد `cd repo && npm install`.",
    },
    {
        "source": "The error says: TypeError: Cannot read properties of undefined (reading 'map').",
        "target": "الخطأ كيقول: TypeError: Cannot read properties of undefined (reading 'map').",
    },
    {
        "source": "Set the environment variable API_KEY=sk-abc123 before running the script.",
        "target": "حط المتغير ديال البيئة API_KEY=sk-abc123 قبل ما تخدّم السكريبت.",
    },
    {
        "source": "The function signature is `def train(model, dataset, epochs=3, lr=5e-5):`.",
        "target": "الشكل ديال الفونكشن هوا `def train(model, dataset, epochs=3, lr=5e-5):`.",
    },
    {
        "source": "The command is `python main.py --batch_size 16 --lr 2e-4`.",
        "target": "الكوموند هيا `python main.py --batch_size 16 --lr 2e-4`.",
    },
    {
        "source": "The error message is: IndexError: list index out of range.",
        "target": "الخطاء لي طرا هوا: IndexError: list index out of range.",
    },
    {
        "source": "Install it with `pip install torch==2.1.0`.",
        "target": "دخلو بـ `pip install torch==2.1.0`.",
    },

    # --- structural / formatting instructions (round 2) ---
    {
        "source": "List the 5 pillars of Islam as a numbered list.",
        "target": "عطيني 5 أركان الإسلام على شكل ليستة مرقمة.",
    },
    {
        "source": "Respond with ONLY 'yes' or 'no'. Nothing else.",
        "target": "جاوب غير بـ 'yes' ولا 'no'. ما تزيد حتى حاجة خرى.",
    },
    {
        "source": "Write a haiku about the ocean.",
        "target": "كتب ليا قصيدة هايكو على البحر.",
    },
    {
        "source": "Give me exactly 3 pros and 3 cons in two separate bullet lists.",
        "target": "عطيني بالضبط 3 ديال المزايا و3 ديال العيوب فجوج لوائح ديال النقط منفصلين.",
    },
    {
        "source": "Summarize this in exactly 2 sentences.",
        "target": "لخّص ليا هادشي فبالضبط 2 ديال الجمل.",
    },
    {
        "source": "Format the output as a JSON object.",
        "target": "عطيني الخارج على شكل JSON object.",
    },

    # --- narrative / 3rd-person tracking (do NOT change person) ---
    {
        "source": "Sarah went to the market. She bought tomatoes, onions, and fresh bread. On her way home, she ran into her neighbor Ahmed. He asked if she could lend him some onions. She gave him three.",
        "target": "سارة مشات للسوق. شرات مطيشة وبصلة وخبز طري. فالطريق للدار، تلاقات مع الجار ديالها أحمد. سولها واش تقدر تسلفو شوية ديال البصلة. عطاتو ثلاثة.",
    },
    {
        "source": "Ali opened the door. He saw his friend Youssef standing outside. Youssef asked him to come play football. Ali said he would come after finishing his homework.",
        "target": "علي حلّ الباب. شاف صاحبو يوسف واقف برّا. يوسف سولو واش يجي يلعب الكورة. علي قال ليه غادي يجي من بعد ما يكمل التمارين.",
    },
    {
        "source": "Fatima cooked dinner for her family. Her children helped set the table. They all sat together and enjoyed the meal.",
        "target": "فاطمة طيبات العشا للعائلة ديالها. ولادها عاونوها حطّو الطابلة. كلهم قعدو مع بعضياتهم وتمتّعو بالماكلة.",
    },

    # --- negation / double negation / subtle meaning ---
    {
        "source": "I never said she stole the money.",
        "target": "ما عمرني قلت بلي هيا سرقات الفلوس.",
    },
    {
        "source": "He is not unhappy with the results.",
        "target": "هوا راضي بالنتائج.",
    },
    {
        "source": "She almost didn't make it to the interview on time.",
        "target": "بشوية وكانت غادية تعطل على الانترفيو.",
    },
    {
        "source": "It's not that I don't like it, I just prefer the other one.",
        "target": "ماشي زعما ماعجبنيش، غير كنفضل لاخور.",
    },
    {
        "source": "She is not unaware of the problem.",
        "target": "هيا واعية بالمشكل.",
    },
    {
        "source": "He was not entirely wrong.",
        "target": "ما كانش غالط كلّياً.",
    },

    # --- conditional / hypothetical ---
    {
        "source": "If I had known about the traffic, I would have left earlier.",
        "target": "كون عرفت بلي كاين الزحام، كون خرجت بكري.",
    },

    # --- quoted speech: preserve quote content faithfully ---
    {
        "source": 'He said, "Meet me at the café at 5 PM."',
        "target": 'قال: "تلاقاني فالقهوة مع 5 ديال لعشية."',
    },
    {
        "source": 'She whispered, "Don\'t tell anyone about this."',
        "target": 'قالت ليه بالسرّ: "ما تقول لحتى واحد على هادشي."',
    },
    {
        "source": 'The teacher said, "Open your books to page 42."',
        "target": 'الأستاذ قال: "حلّو الكتاب فصفحة 42."',
    },

    # --- technical content: do NOT hallucinate meanings ---
    {
        "source": "The status code is 404 Not Found. Please check the URL.",
        "target": "الـ status code هوا 404 Not Found. تأكد من الـ URL.",
    },
    {
        "source": "The HTTP response returned 500 Internal Server Error.",
        "target": "الـ HTTP response رجع 500 Internal Server Error.",
    },
    {
        "source": "The API returned a 401 Unauthorized error.",
        "target": "الـ API رجعات خطأ 401 Unauthorized.",
    },

    # --- hashtags: do NOT translate ---
    {
        "source": "The hashtag #MoroccoTravel is trending on Twitter.",
        "target": "الهاشتاڭ #MoroccoTravel طالع طوندونس فتويتر.",
    },
    {
        "source": "Use the hashtag #OpenSource when you post about it.",
        "target": "استعمل الهاشتاڭ #OpenSource فاش تپوسطي عليه.",
    },
    {
        "source": "The hashtag #AI2025 went viral on social media.",
        "target": "الهاشتاڭ #AI2025 تشهر فسوشل ميديا.",
    },

    # --- passwords / credentials: preserve EXACTLY ---
    {
        "source": "My Wi-Fi password is P@ssw0rd!2024. Don't share it.",
        "target": "الباسوورد ديال الواي فاي ديالي هوا P@ssw0rd!2024. ما تشاركوش.",
    },
    {
        "source": "The access code is XK-7829-QM. Enter it exactly.",
        "target": "كود الدخول هوا XK-7829-QM. دخّلو بالضبط.",
    },

    # --- gender preservation ---
    {
        "source": "She finished her homework before dinner.",
        "target": "كمّلات التمارين ديالها قبل العشا.",
    },
    {
        "source": "She is a talented engineer.",
        "target": "هيا مهندسة موهوبة.",
    },
    {
        "source": "She told him the truth.",
        "target": "قالت ليه الحقيقة.",
    },
    {
        "source": "She ran faster than everyone else.",
        "target": "جرات بالزربة كثر من كلشي.",
    },

    # --- location hallucination prevention ---
    {
        "source": "The conference is in San Francisco, California.",
        "target": "الكونفيرونس فـ San Francisco، كاليفورنيا.",
    },
    {
        "source": "He moved to Austin, Texas last year.",
        "target": "نقل لـ Austin، تيكساس العام اللي فات.",
    },
    {
        "source": "The office is in downtown Toronto, Canada.",
        "target": "المكتب فوسط Toronto، كندا.",
    },
    {
        "source": "She studied at MIT in Cambridge, Massachusetts.",
        "target": "قرات فـ MIT فـ Cambridge، ماساتشوستس.",
    },

    # --- "traffic" = azدحام / bouchon, NOT car breakdown ---
    {
        "source": "There was a lot of traffic on the highway.",
        "target": "كان الزحام بزاف فالأوتوروت.",
    },
    {
        "source": "I was stuck in traffic for two hours.",
        "target": "بقيت حاصل فالزحام ساعتين.",
    },

    # --- respond with exact keywords, don't transliterate ---
    {
        "source": "Reply with 'yes' or 'no' only.",
        "target": "جاوب غير بـ 'yes' ولا 'no'.",
    },
    {
        "source": "Type 'confirm' to proceed.",
        "target": "كتب 'confirm' باش تكمل.",
    },
    {
        "source": "Enter 'quit' to exit the program.",
        "target": "دخّل 'quit' باش تخرج من البروڭرام.",
    },

    # --- greetings & farewells (model confuses hello / goodbye) ---
    {
        "source": "Hello",
        "target": "سلام",
    },
    {
        "source": "Hello, how are you?",
        "target": "سلام، كيداير؟",
    },
    {
        "source": "Hi there!",
        "target": "سلام!",
    },
    {
        "source": "Goodbye",
        "target": "بسلامة",
    },
    {
        "source": "Goodbye, see you tomorrow.",
        "target": "بسلامة، نتقاشعو غدا.",
    },
    {
        "source": "See you later.",
        "target": "نتقاشعو من بعد.",
    },
    {
        "source": "Good morning.",
        "target": "صباح الخير.",
    },
    {
        "source": "Good night.",
        "target": "تصبح على خير.",
    },
    {
        "source": "Good evening.",
        "target": "مسا الخير.",
    },

    # --- bored vs worried ---
    {
        "source": "I'm bored.",
        "target": "جاني الملال.",
    },
    {
        "source": "I'm bored, there's nothing to do.",
        "target": "جاني الملال، ما كاين ما يدار.",
    },
    {
        "source": "I'm so bored at home.",
        "target": "قنطت بزاف فالدار.",
    },
    {
        "source": "I'm worried about the exam.",
        "target": "مشطون مع الامتحان.",
    },
    {
        "source": "I'm worried about my mom.",
        "target": "راني مخلوع على ماما.",
    },

    # --- "No thanks" = لا شكراً (not ما شكراً) ---
    {
        "source": "No thanks.",
        "target": "لا شكراً.",
    },
    {
        "source": "No, thank you.",
        "target": "لا، شكراً.",
    },
    {
        "source": "Yes, please.",
        "target": "ايه، عافاك.",
    },
    {
        "source": "No problem.",
        "target": "ما كاين باس.",
    },

    # --- family terms (Darija, not MSA attached pronouns) ---
    {
        "source": "My grandmother makes the best tagine in the world.",
        "target": "جداتي كتصاوب أحسن طاجين فالعالم.",
    },
    {
        "source": "My grandmother told me a story before bed.",
        "target": "جداتي عاودات ليا خرافة قبل النعاس.",
    },
    {
        "source": "My grandfather used to wake up early.",
        "target": "جدي كان كيفيق بكري.",
    },
    {
        "source": "My mother called me.",
        "target": "ماما عيطات ليا.",
    },
    {
        "source": "My father works in Casablanca.",
        "target": "بابا كيخدم فكازا.",
    },

    # --- taxi ---
    {
        "source": "The taxi driver took the long route and charged me double.",
        "target": "مول الطاكسي دار من الطريق الطويلة وخلصني الدوبل.",
    },
    {
        "source": "I took a taxi to the airport.",
        "target": "شديت طاكسي للمطار.",
    },
    {
        "source": "The taxi is expensive here.",
        "target": "الطاكسي غالي هنا.",
    },
    {
        "source": "The taxi driver was very nice.",
        "target": "مول الطاكسي كان ضريف بزاف.",
    },

    # --- "call me" (عيط ليا ≠ "tell me") ---
    {
        "source": "Call me later.",
        "target": "عيط ليا من بعد.",
    },
    {
        "source": "Call me when you arrive.",
        "target": "عيط ليا فاش توصل.",
    },
    {
        "source": "Don't call me at night.",
        "target": "ما تعيطش ليا فالليل.",
    },
    {
        "source": "He called me yesterday.",
        "target": "عيط ليا البارح.",
    },
    {
        "source": "I'll call you back.",
        "target": "غادي نعيط ليك من بعد.",
    },

    # --- science terms (Ohm = أوم not أومر) ---
    {
        "source": "Ohm's law states that V = IR.",
        "target": "قانون أوم كيقول بلي V = IR.",
    },
    {
        "source": "Newton's first law of motion explains inertia.",
        "target": "القانون الأول ديال نيوتن فالحركة كيشرح الجمود.",
    },

    # --- idioms (round 3 — natural Darija equivalents) ---
    {
        "source": "I'm feeling under the weather today.",
        "target": "حاسس راسي مريض اليوم.",
    },
    {
        "source": "That's water under the bridge now.",
        "target": "هادشي فات ومشى دابا.",
    },
    {
        "source": "He told a white lie to avoid hurting her feelings.",
        "target": "كذب عليها كذبة صغيرة باش ما يجرحش إحساسها.",
    },
    {
        "source": "They are like two peas in a pod.",
        "target": "فولة وتقسمات على جوج.",
    },
    {
        "source": "She hit the ground running at her new job.",
        "target": "بدات الخدمة الجديدة ديالها بنشاط من أول يوم.",
    },
    {
        "source": "He knocked it out of the park with that presentation.",
        "target": "نفض ديك البريزونتاسيون.",
    },
    {
        "source": "Don't cry over spilled milk.",
        "target": "اللي فات مات.",
    },
    {
        "source": "He's burning the candle at both ends.",
        "target": "كيتعب راسو بزاف وما بقاش كيرتاح.",
    },
    {
        "source": "She's over the moon about her promotion.",
        "target": "طارت بالفرحة بالترقية ديالها.",
    },
    {
        "source": "Break a leg at the audition!",
        "target": "الله يسر لك فالأوديسيون!",
    },
    {
        "source": "He spilled the beans about the surprise.",
        "target": "فرش البلان ديال المفاجأة.",
    },
    {
        "source": "You're barking up the wrong tree.",
        "target": "غالط فالعنوان.",
    },
    {
        "source": "She has a heart of gold.",
        "target": "قلبها بيض.",
    },
    {
        "source": "It's a piece of cake.",
        "target": "ساهلة ماهلة.",
    },
    {
        "source": "He's pulling my leg.",
        "target": "كيضحك عليا.",
    },
    {
        "source": "The ball is in your court.",
        "target": "الكورة فالملعب ديالك.",
    },
    {
        "source": "Actions speak louder than words.",
        "target": "الفعل حسن من القول.",
    },
    {
        "source": "Every cloud has a silver lining.",
        "target": "كل توخيرة فيها خيرة.",
    },

    # --- daily life / common short phrases ---
    {
        "source": "I forgot my keys at home.",
        "target": "نسيت السوارت فالدار.",
    },
    {
        "source": "Close the door.",
        "target": "سدّ الباب.",
    },
    {
        "source": "Open the window.",
        "target": "حلّ الشرجم.",
    },
    {
        "source": "Turn off the lights.",
        "target": "طفي الضو.",
    },
    {
        "source": "Wake me up at 7.",
        "target": "فيّقني مع 7.",
    },
    {
        "source": "I'm hungry.",
        "target": "فيا الجوع.",
    },
    {
        "source": "I'm thirsty.",
        "target": "فيا العطش.",
    },
    {
        "source": "I'm tired.",
        "target": "عييت.",
    },
    {
        "source": "I'm cold.",
        "target": "فيا البرد.",
    },
    {
        "source": "I'm cold.",
        "target": "جاني البرد.",
    },
    {
        "source": "I'm hot.",
        "target": "جاني الصهد.",
    },
    {
        "source": "It's raining.",
        "target": "كتصب الشتا.",
    },
    {
        "source": "What time is it?",
        "target": "شحال فالساعة؟",
    },
    {
        "source": "Where is the bathroom?",
        "target": "فين كاين الطواليت؟",
    },
    {
        "source": "How much does this cost?",
        "target": "بشحال هادشي؟",
    },
    {
        "source": "I don't understand.",
        "target": "ما فهمتش.",
    },
    {
        "source": "Can you repeat that?",
        "target": "واش تقدر تعاود؟",
    },
    {
        "source": "I need help.",
        "target": "عاوني عافاك.",
    },
    {
        "source": "Wait for me.",
        "target": "تسناني.",
    },
    {
        "source": "Hurry up!",
        "target": "سربي!",
    },
    {
        "source": "Hurry up!",
        "target": "زرب!",
    },
    {
        "source": "Slow down.",
        "target": "غير بالشوية.",
    },
    {
        "source": "Slow down.",
        "target": "غير بالمهل.",
    },

    # --- gender tracking narratives ---
    {
        "source": "The manager fired his best employee because she was always late.",
        "target": "الباطرون جرى على أحسن موظفة عندو حيت كانت ديما كتعطل.",
    },
    {
        "source": "She is a doctor. Her patients love her because she listens carefully.",
        "target": "هيا طبيبة. المرضى ديالها كيبغيوها حيت كتسمع ليهم مزيان.",
    },
    {
        "source": "He asked his sister for help. She immediately agreed.",
        "target": "طلب من ختو تعاونو. وافقات فالبلاصة.",
    },
    {
        "source": "Khadija passed her driving test on the first try. She was so happy.",
        "target": "خديجة شدات البيرمي كي دوزات الامتحان أول مرة. كانت فرحانة بزاف.",
    },
    {
        "source": "The teacher asked the student a question. He didn't know the answer.",
        "target": "الأستاذ سول التلميذ واحد السؤال. ما عرفش يجاوب.",
    },
    {
        "source": "My sister just graduated. She got a job in Rabat.",
        "target": "ختي يلاه تخرجات. لقات خدمة فالرباط.",
    },

    # --- explicit masculine / feminine mapping ---
    {
        "source": "He is very tired.",
        "target": "هوا عيان بزاف.",
    },
    {
        "source": "She is very tired.",
        "target": "هيا عيانة بزاف.",
    },
    {
        "source": "He was happy when he heard the news.",
        "target": "كان فرحان ملي سمع الخبار.",
    },
    {
        "source": "She was happy when she heard the news.",
        "target": "كانت فرحانة ملي سمعات الخبار.",
    },
    {
        "source": "He is my friend.",
        "target": "هوا صاحبي.",
    },
    {
        "source": "She is my friend.",
        "target": "هيا صاحبتي.",
    },
    {
        "source": "He works at the bank.",
        "target": "هوا خدام فالبنكة.",
    },
    {
        "source": "She works at the bank.",
        "target": "هيا خدامة فالبنكة.",
    },
    {
        "source": "He is looking for a job.",
        "target": "هوا كيقلب على خدمة.",
    },
    {
        "source": "She is looking for a job.",
        "target": "هيا كتقلب على خدمة.",
    },

    # --- "of course" / "naturally" = بطبيعة الحال (not أكيد) ---
    {
        "source": "Of course I'll help you.",
        "target": "بطبيعة الحال غادي نعاونك.",
    },
    {
        "source": "Of course, you're welcome to join us.",
        "target": "بطبيعة الحال، مرحبا بيك معانا.",
    },
    {
        "source": "Naturally, he was the first to arrive.",
        "target": "بطبيعة الحال، هوا كان اللول اللي وصل.",
    },

    # --- DR→EN critical: Darija expressions model must understand ---
    {
        "source": "He nailed it.",
        "target": "جابها لاصقة.",
    },
    {
        "source": "Strike while the iron is hot.",
        "target": "ضرب الحديد وهو سخون.",
    },
    {
        "source": "He's playing with fire.",
        "target": "كيلعب بالعافية.",
    },
    {
        "source": "She's a hard worker.",
        "target": "خدّامة بزاف.",
    },
    {
        "source": "Mind your own business.",
        "target": "ديها فراسك.",
    },
    {
        "source": "It's none of your business.",
        "target": "ماشي شغلك.",
    },
    {
        "source": "He's stingy.",
        "target": "هوا زقرام.",
    },
    {
        "source": "She's generous.",
        "target": "تبارك الله سخية.",
    },

    # --- French loanwords common in Darija ---
    {
        "source": "The bus is late.",
        "target": "الطوبيس معطّل.",
    },
    {
        "source": "I need to charge my phone.",
        "target": "خاصني نشارجي التيليفون ديالي.",
    },
    {
        "source": "The car won't start.",
        "target": "الطوموبيل ما بغاتش تخدم.",
    },
    {
        "source": "Park the car over there.",
        "target": "حط الطوموبيل لهيه.",
    },
    {
        "source": "I have an appointment with the doctor.",
        "target": "عندي رونديفو عند الطبيب.",
    },

    # ================================================================
    # ROUND 4 — MASSIVE EXPANSION
    # ================================================================

    # ── Animals (fixing rooster=الفروج, NOT البيروكي) ──────────────────
    {
        "source": "The rooster crows every morning at dawn.",
        "target": "الفروج كيقوق كلا صباح مع الفجر.",
    },
    {
        "source": "My neighbor's rooster wakes me up every morning at 5 AM.",
        "target": "الفروج ديال جاري كيفيقني كل صباح مع 5.",
    },
    {
        "source": "We have chickens and a rooster in the backyard.",
        "target": "عندنا الدجاج والفروج فالكوري.",
    },
    {
        "source": "The rooster is loud.",
        "target": "الفروج صوتو مجهد بزاف.",
    },
    {
        "source": "The parrot can talk.",
        "target": "البيروكي كيهضر.",
    },
    {
        "source": "I have a parrot at home.",
        "target": "عندي ببغاء فالدار.",
    },
    {
        "source": "The cat is sleeping on the couch.",
        "target": "المش ناعس فالفوتاي.",
    },
    {
        "source": "My cat ran away.",
        "target": "المش ديالي هرب.",
    },
    {
        "source": "The dog is barking.",
        "target": "الكلب كينبح.",
    },
    {
        "source": "There is a stray dog in the street.",
        "target": "كاين كلب ديال زنقا فالزنقة.",
    },
    {
        "source": "I'm afraid of snakes.",
        "target": "كنخاف من الحناش.",
    },
    {
        "source": "There's a fly in my soup.",
        "target": "كاينة دبانة فالحريرة ديالي.",
    },
    {
        "source": "The fish is fresh.",
        "target": "الحوت جديد.",
    },
    {
        "source": "We bought a sheep for Eid.",
        "target": "شرينا حولي للعيد.",
    },
    {
        "source": "We bought a goat for Eid.",
        "target": "شرينا معزة للعيد.",
    },
    {
        "source": "The horse is beautiful.",
        "target": "هاد العود زوين.",
    },
    {
        "source": "There are ants in the kitchen.",
        "target": "كاين النمل فالكوزينة.",
    },
    {
        "source": "The cow gives milk.",
        "target": "البقرة كتعطي الحليب.",
    },
    {
        "source": "The donkey is carrying heavy bags.",
        "target": "الحمار هاز الخناشي ثقال.",
    },
    {
        "source": "I saw a rabbit in the garden.",
        "target": "شفت قنية فالجردة.",
    },
    {
        "source": "The bird is singing.",
        "target": "الفرخ كيغني.",
    },

    # ── Common nouns (fixing wallet, umbrella, etc.) ──────────────────
    {
        "source": "I lost my wallet.",
        "target": "ضيعت البزطام ديالي.",
    },
    {
        "source": "My wallet is empty.",
        "target": "البزطام ديالي خاوي.",
    },
    {
        "source": "I left my wallet in the petit taxi and the driver returned it.",
        "target": "نسيت البزطام فالطاكسي الصغير ومول الطاكسي رجعو ليا.",
    },
    {
        "source": "Where is my wallet?",
        "target": "فين البزطام ديالي؟",
    },
    {
        "source": "Take your umbrella, it might rain.",
        "target": "دي معاك المظل، تقدر تصب الشتا.",
    },
    {
        "source": "I forgot my umbrella at the office.",
        "target": "نسيت المظل فالبيرو.",
    },
    {
        "source": "Do you have an umbrella?",
        "target": "واش عندك مظل؟",
    },
    {
        "source": "The umbrella is broken.",
        "target": "المظل مهرس.",
    },
    {
        "source": "I need a towel.",
        "target": "خاصني فوطة.",
    },
    {
        "source": "Give me the remote control.",
        "target": "عطيني التيليكوموند.",
    },
    {
        "source": "The mirror is dirty.",
        "target": "المراية موسخة.",
    },
    {
        "source": "I bought a new pillow.",
        "target": "شريت مخدة جديدة.",
    },
    {
        "source": "The blanket is warm.",
        "target": "المانطة سخونة.",
    },
    {
        "source": "Hang the clothes on the line.",
        "target": "نشر الحوايج فالحبل.",
    },
    {
        "source": "I need to iron my shirt.",
        "target": "خاصني نصلح القميجة ديالي.",
    },
    {
        "source": "The fridge is empty.",
        "target": "الثلاجة خاوي.",
    },
    {
        "source": "The washing machine is broken.",
        "target": "ماكينة الصابون خاسرة.",
    },
    {
        "source": "Where are my glasses?",
        "target": "فين النداضر ديالي؟",
    },
    {
        "source": "I forgot my bag at school.",
        "target": "نسيت الصاك ديالي فالمدرسة.",
    },
    {
        "source": "The elevator is broken.",
        "target": "السانسور خاسر.",
    },
    {
        "source": "Take the stairs.",
        "target": "طلع فالدروج.",
    },
    {
        "source": "The ceiling fan is noisy.",
        "target": "الفرفارة ديال السقف فيها الصداع.",
    },
    {
        "source": "I need batteries for the remote.",
        "target": "خاصني الحجرات للتيليكوموند.",
    },
    {
        "source": "The lock is jammed.",
        "target": "الساروت وحل.",
    },
    {
        "source": "I can't find the charger.",
        "target": "ما لقيتش الشارجور.",
    },

    # ── Family terms (expanded) ──────────────────────────────────────
    {
        "source": "My uncle lives in Tangier.",
        "target": "عمي ساكن فطنجة.",
    },
    {
        "source": "My maternal uncle came to visit.",
        "target": "خالي جا يزورنا.",
    },
    {
        "source": "My aunt makes great pastries.",
        "target": "عمتي كتصاوب حلوة زوينة.",
    },
    {
        "source": "My maternal aunt lives next door.",
        "target": "خالتي ساكنة حدانا.",
    },
    {
        "source": "The couscous on Friday at my aunt's house is the best tradition ever.",
        "target": "الكسكسو نهار الجمعة عند خالتي هيا أحسن عادة.",
    },
    {
        "source": "My uncle's son got married last month.",
        "target": "ولد عمي تزوج الشهر اللي فات.",
    },
    {
        "source": "My aunt's daughter is studying medicine.",
        "target": "بنت خالتي كتقرا الطب.",
    },
    {
        "source": "My brother-in-law works in France.",
        "target": "نسيبي كيخدم ففرانسا.",
    },
    {
        "source": "My sister-in-law is pregnant.",
        "target": "نسيبتي حاملة.",
    },
    {
        "source": "My son started school this year.",
        "target": "ولدي بدا يقرا هاد العام.",
    },
    {
        "source": "My daughter passed her exam.",
        "target": "بنتي نجحات فالامتحان.",
    },
    {
        "source": "My wife is a teacher.",
        "target": "مراتي أستاذة.",
    },
    {
        "source": "My husband travels a lot for work.",
        "target": "راجلي كيسافر بزاف على ود الخدمة.",
    },
    {
        "source": "My parents are getting old.",
        "target": "الوالدين ديالي كبرو.",
    },
    {
        "source": "I miss my family.",
        "target": "توحشت العائلة ديالي.",
    },
    {
        "source": "My nephew is very smart.",
        "target": "ولد ختي ذكي بزاف.",
    },
    {
        "source": "My niece is adorable.",
        "target": "بنت خويا زوينة بزاف.",
    },
    {
        "source": "My mother-in-law cooks delicious food.",
        "target": "النسيبة ديالي كتطيب ماكلة بنينة.",
    },
    {
        "source": "My father-in-law is retired.",
        "target": "النسيب ديالي متقاعد.",
    },
    {
        "source": "My twin brothers look exactly alike.",
        "target": "التوام ديالي كيتشابهو ديال بصح.",
    },

    # ── Body parts ────────────────────────────────────────────────────
    {
        "source": "My head hurts.",
        "target": "كيضرني راسي.",
    },
    {
        "source": "I have a stomachache.",
        "target": "كتضرني كرشي.",
    },
    {
        "source": "My back is killing me.",
        "target": "ضهري كيقتلني.",
    },
    {
        "source": "I hurt my knee.",
        "target": "ضربت الركبة ديالي.",
    },
    {
        "source": "He broke his arm.",
        "target": "تهرسات ليه يدو.",
    },
    {
        "source": "She twisted her ankle.",
        "target": "تلوات ليها رجليها.",
    },
    {
        "source": "My tooth hurts.",
        "target": "كتضرني ضرستي.",
    },
    {
        "source": "His shoulder is sore.",
        "target": "كتفو كيضرو.",
    },
    {
        "source": "I have a sore throat.",
        "target": "كيضرني حلقي.",
    },
    {
        "source": "Wash your hands.",
        "target": "غسل يديك.",
    },
    {
        "source": "My eyes are tired.",
        "target": "عينيا عياو.",
    },
    {
        "source": "My ear is ringing.",
        "target": "ودني كتصفر.",
    },
    {
        "source": "My feet are cold.",
        "target": "رجليا باردين.",
    },
    {
        "source": "He has a big nose.",
        "target": "نيفو كبير.",
    },
    {
        "source": "Her hair is long.",
        "target": "شعرها طويل.",
    },

    # ── Food & cooking ────────────────────────────────────────────────
    {
        "source": "I need to buy groceries.",
        "target": "خاصني نتقضى.",
    },
    {
        "source": "I went to the grocery store.",
        "target": "مشيت للحانوت.",
    },
    {
        "source": "I need tomatoes, onions, olive oil, and fresh mint.",
        "target": "خاصني مطيشة والبصلة وزيت الزيتون والنعناع طري.",
    },
    {
        "source": "Add salt and pepper.",
        "target": "زيد الملحة والإبزار.",
    },
    {
        "source": "The bread is stale.",
        "target": "الخبز يابس.",
    },
    {
        "source": "The soup is too hot.",
        "target": "الحريرة سخونة بزاف.",
    },
    {
        "source": "She's making couscous.",
        "target": "كتصاوب الكسكسو.",
    },
    {
        "source": "The tagine smells amazing.",
        "target": "الطاجين ريحتو زوينة بزاف.",
    },
    {
        "source": "I want tea with mint.",
        "target": "بغيت أتاي بالنعناع.",
    },
    {
        "source": "The coffee is bitter.",
        "target": "القهوة مرة.",
    },
    {
        "source": "Do you want sugar in your tea?",
        "target": "واش بغيتي السكار فالأتاي؟",
    },
    {
        "source": "The chicken is not cooked yet.",
        "target": "الدجاج مازال ما طاب.",
    },
    {
        "source": "I'm allergic to peanuts.",
        "target": "عندي الحساسية من الكاوكاو.",
    },
    {
        "source": "The watermelon is sweet.",
        "target": " هاد الدلاح حلو.",
    },
    {
        "source": "Peel the potatoes.",
        "target": "قشر البطاطا.",
    },
    {
        "source": "The eggs are boiled.",
        "target": "البيض مسلوق.",
    },
    {
        "source": "I ate too much.",
        "target": "كليت بزاف.",
    },
    {
        "source": "The meat is tough.",
        "target": "اللحم قاسح.",
    },
    {
        "source": "I want a glass of water.",
        "target": "بغيت كاس ديال الما.",
    },
    {
        "source": "The orange juice is fresh.",
        "target": "العصير ديال الليمون طري.",
    },
    {
        "source": "She's baking a cake.",
        "target": "كتصاوب الكيكة.",
    },
    {
        "source": "The olive oil is from Morocco.",
        "target": "زيت الزيتون من المغرب.",
    },
    {
        "source": "I'm fasting today.",
        "target": "أنا صايم اليوم.",
    },
    {
        "source": "Breakfast is ready.",
        "target": "الفطور واجد.",
    },
    {
        "source": "What's for lunch?",
        "target": "شنو كاين فالغدا؟",
    },
    {
        "source": "Dinner is late today.",
        "target": "العشا معطل اليوم.",
    },

    # ── Weather ───────────────────────────────────────────────────────
    {
        "source": "It's very hot today.",
        "target": "الصهد بزاف اليوم.",
    },
    {
        "source": "It's cold outside.",
        "target": "بارد الحال برا.",
    },
    {
        "source": "The wind is strong.",
        "target": "الريح مجهد.",
    },
    {
        "source": "It's snowing in the mountains.",
        "target": "كيطيح الثلج فالجبال.",
    },
    {
        "source": "There's fog on the road.",
        "target": "كاين الضبابة فالطريق.",
    },
    {
        "source": "The sun is burning.",
        "target": "الشمس كتحرق.",
    },
    {
        "source": "It stopped raining.",
        "target": "وقفات الشتا.",
    },
    {
        "source": "The weather is nice today.",
        "target": "الجو زوين اليوم.",
    },
    {
        "source": "It's humid today.",
        "target": "الرطوبة بزاف اليوم.",
    },
    {
        "source": "There might be a storm tonight.",
        "target": "يمكن تكون عاصفة الليلة.",
    },
    {
        "source": "The electricity went out during the storm last night.",
        "target": "الضو تقطع فالعاصفة البارح بالليل.",
    },
    {
        "source": "The power is out.",
        "target": "الضو مقطوع.",
    },

    # ── Colors ────────────────────────────────────────────────────────
    {
        "source": "The car is red.",
        "target": "الطوموبيل حمرا.",
    },
    {
        "source": "I want the blue one.",
        "target": "بغيت الزرقة.",
    },
    {
        "source": "The sky is gray today.",
        "target": "السما مغيمة اليوم.",
    },
    {
        "source": "She's wearing a white dress.",
        "target": "لابسة كسوة بيضا.",
    },
    {
        "source": "The walls are yellow.",
        "target": "الحيوط صفرين.",
    },
    {
        "source": "He bought a black jacket.",
        "target": "شرا جاكيطة كحلة.",
    },
    {
        "source": "The green one is cheaper.",
        "target": "الخضرة رخيصة كثر.",
    },

    # ── Directions & locations ────────────────────────────────────────
    {
        "source": "Turn right at the traffic light.",
        "target": "دور على اليمين عند الضو لحمر.",
    },
    {
        "source": "Turn left after the mosque.",
        "target": "دور على اليسار من بعد الجامع.",
    },
    {
        "source": "Go straight ahead.",
        "target": "سير نيشان.",
    },
    {
        "source": "It's on the corner.",
        "target": "فالقنت.",
    },
    {
        "source": "The school is near the market.",
        "target": "المدرسة قريبة للسوق.",
    },
    {
        "source": "The hospital is far from here.",
        "target": "السبيطار بعيد من هنا.",
    },
    {
        "source": "Where is the nearest pharmacy?",
        "target": "فين أقرب فارماسيان؟",
    },
    {
        "source": "The pharmacy is closed.",
        "target": "الفرماسيان مسدود.",
    },
    {
        "source": "Cross the street carefully.",
        "target": "قطع الشارع بشوية.",
    },
    {
        "source": "The building is on the left side.",
        "target": "العمارة فالجيهة ديال ليسر.",
    },
    {
        "source": "I'm lost.",
        "target": "تلفت.",
    },
    {
        "source": "Can you show me on the map?",
        "target": "واش تقدر توريني فالخريطة؟",
    },

    # ── Transport (expanded) ──────────────────────────────────────────
    {
        "source": "The bus is late.",
        "target": "الطوبيس معطّل.",
    },
    {
        "source": "I missed the bus.",
        "target": "فاتني الطوبيس.",
    },
    {
        "source": "The train to Tangier leaves at 9 AM.",
        "target": "التران ديال طنجة كيمشي مع 9 ديال الصباح.",
    },
    {
        "source": "I took a grand taxi to Rabat.",
        "target": "ركبت فالطاكسي الكبير للرباط.",
    },
    {
        "source": "The tram is crowded.",
        "target": "الطرامواي عامر.",
    },
    {
        "source": "I need to renew my driver's license.",
        "target": "خاصني نجدد البيرمي.",
    },
    {
        "source": "The gas station is 2 kilometers from here.",
        "target": "البومبا ديال ليسانس راها جوج كيلومتر من هنا.",
    },
    {
        "source": "My tire is flat.",
        "target": "الرويضة ديالي تفشات.",
    },
    {
        "source": "I got a parking ticket.",
        "target": "خديت مخالفة ديال الباركينغ.",
    },
    {
        "source": "The highway is jammed.",
        "target": "لوطوروت عامرة.",
    },
    {
        "source": "There was a car accident.",
        "target": "وقعات كسيدة.",
    },
    {
        "source": "The plane is delayed.",
        "target": "الطيارة تعطلات.",
    },

    # ── School & education ────────────────────────────────────────────
    {
        "source": "I have homework to finish.",
        "target": "عندي التمارين خاصني نكملهم.",
    },
    {
        "source": "The teacher explained the lesson well.",
        "target": "الأستاذ شرح الدرس مزيان.",
    },
    {
        "source": "She got a good grade.",
        "target": "خدات نقطة مزيانة.",
    },
    {
        "source": "He failed the exam.",
        "target": "سقط فالامتحان.",
    },
    {
        "source": "School starts at 8:30.",
        "target": "المدرسة كتبدا مع 8:30.",
    },
    {
        "source": "Summer vacation starts next week.",
        "target": "العطلة ديال الصيف كتبدا السيمانة الجاية.",
    },
    {
        "source": "He's studying for the baccalaureate.",
        "target": "كيقرا للباك.",
    },
    {
        "source": "She got accepted into university.",
        "target": "تقبلات فالجامعة.",
    },
    {
        "source": "The exam was difficult.",
        "target": "الامتحان كان صعيب.",
    },
    {
        "source": "I need a notebook and a pen.",
        "target": "خاصني دفتر وستيلو.",
    },
    {
        "source": "The classroom is on the second floor.",
        "target": "القسم فالطابق الثاني.",
    },
    {
        "source": "The students are on break.",
        "target": "التلاميذ عندهوم الاستراحة.",
    },

    # ── Work & employment ─────────────────────────────────────────────
    {
        "source": "I'm looking for a job.",
        "target": "كنقلب على خدمة.",
    },
    {
        "source": "She got fired.",
        "target": "جراو عليها من الخدمة.",
    },
    {
        "source": "He got a raise.",
        "target": "زادوه فالخلصة.",
    },
    {
        "source": "The salary is low.",
        "target": "الخلصة قليل.",
    },
    {
        "source": "I work night shifts.",
        "target": "كنخدم الليل.",
    },
    {
        "source": "My boss is strict.",
        "target": "الباطرون ديالي قاصح.",
    },
    {
        "source": "I have a meeting at 10.",
        "target": "عندي اجتماع مع 10.",
    },
    {
        "source": "She's on maternity leave.",
        "target": "فعطلة الأمومة.",
    },
    {
        "source": "I need to finish this report.",
        "target": "خاصني نكمل هاد الرابور.",
    },
    {
        "source": "The deadline is tomorrow.",
        "target": "اخر أجل غدا.",
    },
    {
        "source": "I'm retired.",
        "target": "أنا متقاعد.",
    },
    {
        "source": "He works at a restaurant.",
        "target": "كيخدم فريسطو.",
    },
    {
        "source": "She's a nurse.",
        "target": "هيا فرملية.",
    },
    {
        "source": "He's an engineer.",
        "target": "هوا مهندس.",
    },
    {
        "source": "She's a lawyer.",
        "target": "هيا محامية.",
    },

    # ── Health & medical ──────────────────────────────────────────────
    {
        "source": "I need to see a doctor.",
        "target": "خاصني نمشي للطبيب.",
    },
    {
        "source": "I have a fever.",
        "target": "فيا السخانة.",
    },
    {
        "source": "Take this medicine twice a day.",
        "target": "خود هاد الدوا جوج مرات فالنهار.",
    },
    {
        "source": "I need cough syrup.",
        "target": "خاصني السيرو ديال الكحة.",
    },
    {
        "source": "He was taken to the hospital.",
        "target": "داوه للسبيطار.",
    },
    {
        "source": "She needs surgery.",
        "target": "خاصها تدير العملية.",
    },
    {
        "source": "I feel dizzy.",
        "target": "جاتني الدوخة.",
    },
    {
        "source": "He fainted.",
        "target": "سخف راه.",
    },
    {
        "source": "I'm on a diet.",
        "target": "أنا داير الريجيم.",
    },
    {
        "source": "He has diabetes.",
        "target": "عندو السكار.",
    },
    {
        "source": "My blood pressure is high.",
        "target": "التونسيون ديالي طالع.",
    },
    {
        "source": "The dentist pulled my tooth.",
        "target": "طبيب السنان حيد ليا الضرصة.",
    },
    {
        "source": "I need to get my eyes checked.",
        "target": "خاصني ندوز على عيني.",
    },
    {
        "source": "She's pregnant.",
        "target": "هيا حاملة.",
    },
    {
        "source": "The baby is due next month.",
        "target": "البيبي غادي يجي الشهر الجاي.",
    },

    # ── Emotions & states (expanded) ──────────────────────────────────
    {
        "source": "I'm happy.",
        "target": "أنا فرحان.",
    },
    {
        "source": "I'm sad.",
        "target": "أنا مقلق.",
    },
    {
        "source": "I'm angry.",
        "target": "أنا معصب.",
    },
    {
        "source": "She's scared.",
        "target": "هيا خايفة.",
    },
    {
        "source": "He's stressed.",
        "target": "هوا مبرزط.",
    },
    {
        "source": "I'm excited.",
        "target": "أنا فرحان ومتحمس.",
    },
    {
        "source": "I'm disappointed.",
        "target": "أنا طاح الضن ديالي.",
    },
    {
        "source": "He's jealous.",
        "target": "هوا محساد.",
    },
    {
        "source": "She's proud of her son.",
        "target": "هيا فرحانة بولدها.",
    },
    {
        "source": "I'm embarrassed.",
        "target": "أنا حشمان.",
    },
    {
        "source": "He's lonely.",
        "target": "هوا بوحدو.",
    },
    {
        "source": "She's annoyed.",
        "target": "هيا راها مقلقة.",
    },
    {
        "source": "I'm confused.",
        "target": "أنا دخت.",
    },
    {
        "source": "He's in a good mood.",
        "target": "هوا فحالة مزيانة.",
    },
    {
        "source": "She's in a bad mood.",
        "target": "معندهاش الجانة اليوم.",
    },

    # ── Shopping ──────────────────────────────────────────────────────
    {
        "source": "How much does this cost?",
        "target": "بشحال هادشي؟",
    },
    {
        "source": "That's too expensive.",
        "target": "غالي بزاف.",
    },
    {
        "source": "Can you give me a discount?",
        "target": "واش تقدر تنقصلي شوية؟",
    },
    {
        "source": "I'm just looking.",
        "target": "غير كنشوف.",
    },
    {
        "source": "Do you have this in a larger size?",
        "target": "واش عندكم هادي فالقياس الكبير؟",
    },
    {
        "source": "I want to return this item.",
        "target": "بغيت نرجع هاد العيبة.",
    },
    {
        "source": "The store closes at 9 PM.",
        "target": "الحانوت كيسد مع 9 ديال لعشية.",
    },
    {
        "source": "I paid with cash.",
        "target": "خلصت بالكاش.",
    },
    {
        "source": "I paid by card.",
        "target": "خلصت بالكارطة.",
    },
    {
        "source": "Give me a bag, please.",
        "target": "عطيني شي ساشي عافاك.",
    },
    {
        "source": "This is a good deal.",
        "target": "هادي همزة زوينة.",
    },
    {
        "source": "The market is closed on Sundays.",
        "target": "السوق مسدود نهار الحد.",
    },

    # ── Time expressions ──────────────────────────────────────────────
    {
        "source": "I'll be there in five minutes.",
        "target": "غادي نوصل فخمسة دقايق.",
    },
    {
        "source": "He's always late.",
        "target": "ديما كيتعطل.",
    },
    {
        "source": "Come early tomorrow.",
        "target": "جي بكري غدا.",
    },
    {
        "source": "I woke up late this morning.",
        "target": "فقت معطل هاد الصباح.",
    },
    {
        "source": "It's still early.",
        "target": "مازال بكري الحال.",
    },
    {
        "source": "It's already midnight.",
        "target": "ديجا نص الليل.",
    },
    {
        "source": "I haven't seen him in a long time.",
        "target": "ما شفتوش مدة طويلة.",
    },
    {
        "source": "The day before yesterday.",
        "target": "والبارح.",
    },
    {
        "source": "The day after tomorrow.",
        "target": "بعد غدا.",
    },
    {
        "source": "Every now and then.",
        "target": "مرة مرة.",
    },
    {
        "source": "Right now.",
        "target": "دابا.",
    },
    {
        "source": "A moment ago.",
        "target": "غير دابا.",
    },
    {
        "source": "Next year.",
        "target": "العام الجاي.",
    },
    {
        "source": "Last year.",
        "target": "العام اللي فات.",
    },

    # ── Idioms (ROUND 4 — new, not in previous rounds) ────────────────
    {
        "source": "Stop beating around the bush.",
        "target": "باراكا من اللف والدوران.",
    },
    {
        "source": "Get to the point.",
        "target": "جيني نيشان.",
    },
    {
        "source": "Stop beating around the bush and tell me what happened.",
        "target": "باراكا من اللف والدوران وقول ليا شنو وقع.",
    },
    {
        "source": "She was walking on eggshells around her boss all week.",
        "target": "كانت كتمشى على البيض سيمانة كولها مع الباطرون ديالها.",
    },
    {
        "source": "He's got a chip on his shoulder about not getting promoted.",
        "target": "ما عجبوش الحال حيت ما ترقاش.",
    },
    {
        "source": "We need to get the ball rolling on this project.",
        "target": "خاصنا نبداو نخدمو على هاد لبروجي.",
    },
    {
        "source": "He burned his bridges when he quit without notice.",
        "target": "خسر كلشي ملي مشى بلا ما يعلم.",
    },
    {
        "source": "She's sitting on the fence about whether to accept the offer.",
        "target": "مازال متحيرة واش تقبل العرض.",
    },
    {
        "source": "He's in hot water with his boss.",
        "target": "عندو مشاكل مع الباطرون ديالو.",
    },
    {
        "source": "She passed with flying colors.",
        "target": "نجحات بامتياز.",
    },
    {
        "source": "That's the last straw.",
        "target": "هادي هيا اللخرة صافي.",
    },
    {
        "source": "He's a jack of all trades.",
        "target": "كيعرف يدير كلشي.",
    },
    {
        "source": "She went the extra mile.",
        "target": "عطات كثر من جهدها.",
    },
    {
        "source": "It's a blessing in disguise.",
        "target": "هادا رزق كان مخبي.",
    },
    {
        "source": "He has a sweet tooth.",
        "target": "كيموت على الحلاوة.",
    },
    {
        "source": "She's got butterflies in her stomach.",
        "target": "شداتها الخلعة من كرشها.",
    },
    {
        "source": "The early bird catches the worm.",
        "target": "اللي فاق بكري ربح.",
    },
    {
        "source": "Rome wasn't built in a day.",
        "target": "دقة دقة كيحمل الواد.",
    },
    {
        "source": "Better late than never.",
        "target": "يجي معطل حسن من ما يجيش فخطرا.",
    },
    {
        "source": "He let the opportunity slip through his fingers.",
        "target": "خلا الفرصة تطيح من يديه.",
    },
    {
        "source": "She's biting off more than she can chew.",
        "target": "دخلات فشي حاجة كبر منها.",
    },
    {
        "source": "Don't put all your eggs in one basket.",
        "target": "ما ديرش البيض ديالك كامل فسلة وحدة.",
    },
    {
        "source": "He's blowing things out of proportion.",
        "target": "كيكبر الحوايج.",
    },
    {
        "source": "She turned a blind eye to the problem.",
        "target": "دارت عين ميكة على المشكل.",
    },
    {
        "source": "We see eye to eye on this.",
        "target": "متفاقين على هادشي.",
    },
    {
        "source": "He got cold feet before the wedding.",
        "target": "شداتو الخلعة قبل العرس.",
    },
    {
        "source": "It cost me an arm and a leg.",
        "target": "تقامت عليا غالية بزاف.",
    },
    {
        "source": "She has a heart of stone.",
        "target": "قلبها قاسح.",
    },
    {
        "source": "Curiosity killed the cat.",
        "target": "اللي دخل في اللي ما يعنيه يسمع اللي ما يرضيه.",
    },
    {
        "source": "Don't bite the hand that feeds you.",
        "target": "ما تعضش اليد اللي كتوكلك.",
    },

    # ── Proverbs (Darija-natural equivalents) ─────────────────────────
    {
        "source": "Early to bed, early to rise.",
        "target": "اللي بكر ربح.",
    },
    {
        "source": "Practice makes perfect.",
        "target": "التكرار كيعلم.",
    },
    {
        "source": "Patience is a virtue.",
        "target": "الصبر مفتاح الفرج.",
    },
    {
        "source": "He who laughs last, laughs best.",
        "target": "اللي كيضحك لخر كيضحك مزيان.",
    },
    {
        "source": "All that glitters is not gold.",
        "target": "ماشي كلشي لي كيلمع هوا الذهب.",
    },
    {
        "source": "A friend in need is a friend indeed.",
        "target": "الصديق وقت الضيق.",
    },
    {
        "source": "Where there's a will, there's a way.",
        "target": "اللي بغى شي حاجة لقى ليها الطريق.",
    },

    # ── Moroccan-specific vocabulary & situations ─────────────────────
    {
        "source": "I went to the hammam yesterday.",
        "target": "مشيت للحمام البارح.",
    },
    {
        "source": "There's a wedding in the neighborhood today.",
        "target": "كاين عرس فالحومة اليوم.",
    },
    {
        "source": "The medina is very crowded.",
        "target": "المدينة القديمة عامرة بزاف.",
    },
    {
        "source": "She's wearing a caftan.",
        "target": "هيا لابسة القفطان.",
    },
    {
        "source": "He bought a djellaba for Eid.",
        "target": "هو شرا جلابة للعيد.",
    },
    {
        "source": "The souk is full of people.",
        "target": "السوق عامر بالناس.",
    },
    {
        "source": "I drink mint tea every afternoon.",
        "target": "كنشرب أتاي بالنعناع كلا عشية.",
    },
    {
        "source": "The call to prayer just sounded.",
        "target": "غير دابا أدن.",
    },
    {
        "source": "Ramadan starts next week.",
        "target": "رمضان كيبدا الصيمانة الجاية.",
    },
    {
        "source": "She went to the public bath.",
        "target": "مشات للحمّام.",
    },
    {
        "source": "I need to exchange money.",
        "target": "خاصني نصرف الفلوس.",
    },
    {
        "source": "The riad is beautiful.",
        "target": "هاد الرياض زوين بزاف.",
    },
    {
        "source": "Let's take a walk in the old city.",
        "target": "يالله نتمشاو فالمدينة القديمة.",
    },
    {
        "source": "He's a street vendor.",
        "target": "هوا بائع متجول.",
    },
    {
        "source": "The barber is closed today.",
        "target": "الحلاق مسدود اليوم.",
    },

    # ── House & home ──────────────────────────────────────────────────
    {
        "source": "The roof is leaking.",
        "target": "السقف كدخل الما.",
    },
    {
        "source": "The kitchen sink is clogged.",
        "target": "لافابو ديال الكوزينة مخنوق.",
    },
    {
        "source": "I need to paint the walls.",
        "target": "خاصني نصبغ الحيوط.",
    },
    {
        "source": "The garden needs watering.",
        "target": "الجردة خاصها السقيان.",
    },
    {
        "source": "The neighbors are noisy.",
        "target": "الجيران مصدعينا بزاف.",
    },
    {
        "source": "The rent is due.",
        "target": "الكرا وصل الوقت ديالو.",
    },
    {
        "source": "I'm moving to a new apartment.",
        "target": "غادي نرحل لشي برطمة جديدة.",
    },
    {
        "source": "The plumber is coming tomorrow.",
        "target": "البلومبي غادي يجي غدا.",
    },
    {
        "source": "The door is stuck.",
        "target": "الباب وحل.",
    },
    {
        "source": "The floor is slippery.",
        "target": "الأرض كتزلق.",
    },
    {
        "source": "Lock the door when you leave.",
        "target": "سد الباب ملي تخرج.",
    },
    {
        "source": "The garbage truck comes in the morning.",
        "target": "الكاميو ديال الزبل كيجي فالصباح.",
    },

    # ── Money & finances ──────────────────────────────────────────────
    {
        "source": "I need to go to the bank.",
        "target": "خاصني نمشي للبنكة.",
    },
    {
        "source": "I need to withdraw money.",
        "target": "خاصني نخرج الفلوس.",
    },
    {
        "source": "The ATM is out of order.",
        "target": "الكيشي خاسر.",
    },
    {
        "source": "I need to pay the electricity bill.",
        "target": "خاصني نخلص الورقة د الضو.",
    },
    {
        "source": "I'm saving money for a car.",
        "target": "كنجمع الفلوس باش نشري طوموبيل.",
    },
    {
        "source": "He owes me 500 dirhams.",
        "target": "كنسالو 500 درهم.",
    },
    {
        "source": "I lent him money.",
        "target": "سلفتو الفلوس.",
    },
    {
        "source": "She borrowed money from me.",
        "target": "تسلفات مني الفلوس.",
    },
    {
        "source": "The prices went up.",
        "target": "الأثمنة طلعو.",
    },
    {
        "source": "Everything is expensive these days.",
        "target": "كلشي غالي هاد الأيام.",
    },

    # ── Phone / technology ────────────────────────────────────────────
    {
        "source": "My phone is dead.",
        "target": "التيليفون ديالي طفا.",
    },
    {
        "source": "I need to charge my phone.",
        "target": "خاصني نشارجي التيليفون.",
    },
    {
        "source": "Send me a message on WhatsApp.",
        "target": "صيفط ليا ميساج فالواتساپ.",
    },
    {
        "source": "The internet is slow.",
        "target": "الكونيكسيون تقيلة.",
    },
    {
        "source": "I can't connect to the WiFi.",
        "target": "ما بغاش يتكونيكطا ليا الويفي.",
    },
    {
        "source": "My screen is cracked.",
        "target": "ليكرون ديالي مهرسة.",
    },
    {
        "source": "I downloaded the app.",
        "target": "طيليشارجيت لابليكاسيون.",
    },
    {
        "source": "The computer is frozen.",
        "target": "البيسي تبلوكا.",
    },
    {
        "source": "I need a new SIM card.",
        "target": "خاصني كارطة جديدة.",
    },
    {
        "source": "My laptop is old.",
        "target": "البيسي بورطابل ديالي قديم.",
    },
    {
        "source": "Take a screenshot.",
        "target": "دير سكرينشوت.",
    },
    {
        "source": "I'll send you the photo.",
        "target": "غادي نصيفط ليك التصويرة.",
    },

    # ── Religion / spiritual ──────────────────────────────────────────
    {
        "source": "God willing.",
        "target": "إن شاء الله.",
    },
    {
        "source": "Thank God.",
        "target": "الحمد لله.",
    },
    {
        "source": "May God protect you.",
        "target": "الله يحفظك.",
    },
    {
        "source": "May God have mercy on him.",
        "target": "الله يرحمو.",
    },
    {
        "source": "May God heal her.",
        "target": "الله يشافيها.",
    },
    {
        "source": "Congratulations!",
        "target": "مبروك!",
    },
    {
        "source": "May God bless you.",
        "target": "الله يبارك فيك.",
    },
    {
        "source": "Prayers are at the mosque.",
        "target": "الصلاة فالجامع.",
    },

    # ── Social expressions & politeness ───────────────────────────────
    {
        "source": "Welcome!",
        "target": "مرحبا!",
    },
    {
        "source": "Nice to meet you.",
        "target": "متشرفين.",
    },
    {
        "source": "How are you doing?",
        "target": "كيداير؟",
    },
    {
        "source": "I'm fine, thank God.",
        "target": "لاباس الحمد لله.",
    },
    {
        "source": "How is your family?",
        "target": "كيدايرة العائلة؟",
    },
    {
        "source": "Excuse me.",
        "target": "سمح ليا.",
    },
    {
        "source": "Sorry.",
        "target": "سمح ليا.",
    },
    {
        "source": "Thank you very much.",
        "target": "شكراً بزاف.",
    },
    {
        "source": "You're welcome.",
        "target": "العفو.",
    },
    {
        "source": "I appreciate it.",
        "target": "شكرا بزاف.",
    },
    {
        "source": "Have a good day.",
        "target": "نهارك مبروك.",
    },
    {
        "source": "Safe travels.",
        "target": "طريق السلامة.",
    },
    {
        "source": "Get well soon.",
        "target": "الله يشافيك.",
    },
    {
        "source": "My condolences.",
        "target": "البركة فراسك.",
    },
    {
        "source": "Happy birthday!",
        "target": "عيد ميلاد سعيد!",
    },
    {
        "source": "Happy Eid!",
        "target": "عيد مبارك سعيد!",
    },

    # ── Personality / character descriptions ──────────────────────────
    {
        "source": "She's generous.",
        "target": "هيا سخية.",
    },
    {
        "source": "He's stingy.",
        "target": "هوا زقرام.",
    },
    {
        "source": "She's patient.",
        "target": "هيا صبّارة.",
    },
    {
        "source": "He's lazy.",
        "target": "هوا معجاز.",
    },
    {
        "source": "She's smart.",
        "target": "هيا ذكية.",
    },
    {
        "source": "He's honest.",
        "target": "هوا عندو الحق.",
    },
    {
        "source": "She's funny.",
        "target": "هيا كضحك.",
    },
    {
        "source": "He's stubborn.",
        "target": "هوا راسو قاسح.",
    },
    {
        "source": "She's kind.",
        "target": "هيا قلبها بيض.",
    },
    {
        "source": "He's rude.",
        "target": "هوا قليل الترابي.",
    },
    {
        "source": "She's shy.",
        "target": "هيا كتحشم.",
    },
    {
        "source": "He's brave.",
        "target": "هوا شجاع.",
    },
    {
        "source": "She's talkative.",
        "target": "فيها كثرة الهضرة.",
    },
    {
        "source": "He's quiet.",
        "target": "هوا سكوتي.",
    },
    {
        "source": "She's hardworking.",
        "target": "هيا كتخدم ديال بصح.",
    },

    # ── Actions / verbs in context ────────────────────────────────────
    {
        "source": "I'm cooking dinner.",
        "target": "كنطيب العشا.",
    },
    {
        "source": "She's cleaning the house.",
        "target": "هيا كتخمل الدار.",
    },
    {
        "source": "He's fixing the car.",
        "target": "هوا كيصلح الطوموبيل.",
    },
    {
        "source": "I'm watching TV.",
        "target": "كنتفرج فالتيليفزيون.",
    },
    {
        "source": "She's reading a book.",
        "target": "هيا كتقرا كتاب.",
    },
    {
        "source": "He's playing football.",
        "target": "هوا كيلعب الكورة .",
    },
    {
        "source": "I'm writing a letter.",
        "target": "كنكتب برية.",
    },
    {
        "source": "She's driving to work.",
        "target": "هيا كتسوق للخدمة.",
    },
    {
        "source": "He's swimming in the pool.",
        "target": "هوا كيعوم فالبيسين.",
    },
    {
        "source": "I'm listening to music.",
        "target": "كنسمع الموسيقى.",
    },
    {
        "source": "She's singing.",
        "target": "هيا كتغني.",
    },
    {
        "source": "He's running.",
        "target": "هوا كيجري.",
    },
    {
        "source": "I'm sleeping.",
        "target": "كنعس.",
    },
    {
        "source": "She's talking on the phone.",
        "target": "كتهضر فالتيليفون.",
    },
    {
        "source": "He's painting the wall.",
        "target": "كيصبغ الحيط.",
    },
    {
        "source": "I'm waiting for the bus.",
        "target": "كنتسنى الطوبيس.",
    },
    {
        "source": "She's searching for her keys.",
        "target": "كتقلب على السوارت ديالها.",
    },
    {
        "source": "He's carrying heavy boxes.",
        "target": "كيهز كراتن ثقال.",
    },
    {
        "source": "I'm packing my suitcase.",
        "target": "كنصايب الباليزة ديالي.",
    },
    {
        "source": "She's watering the plants.",
        "target": "كتسقي الورد.",
    },

    # ── Narrative paragraphs (3rd person, coherent) ───────────────────
    {
        "source": "Layla forgot her umbrella at the office. When she realized, it was already raining. Her colleague Rachid offered to drive her home.",
        "target": "ليلى نسات المظل فالبيرو. ملي فاقت، كانت الشتا ديجا كتصب. زميلها رشيد عرض عليها يوصلها للدار.",
    },
    {
        "source": "Omar asked his wife to pick up the kids from school. She called him back and said she was stuck in a meeting.",
        "target": "عمر طلب من مراتو تجيب الدراري من المدرسة. عيطات ليه و قالت ليه بلي معطلة فاجتماع.",
    },
    {
        "source": "The nurse checked the patient's blood pressure. She told him everything looked normal.",
        "target": "الفرملية عبرات التونسيون ديال المريض. قالت ليه كلشي عادي.",
    },
    {
        "source": "Hassan and his brother Karim opened a bakery together. Their mother was their first customer.",
        "target": "حسن وخوه كريم حلو مخبزة مع بعضياتهم. ماماهم كانت هيا أول كليان عندهم.",
    },
    {
        "source": "Meryem woke up early and made breakfast for her kids. She drove them to school and went to work.",
        "target": "مريم فاقت بكري وصاوبات الفطور للدراري. وصلاتهم للمدرسة ومشات للخدمة.",
    },
    {
        "source": "Youssef was late for the interview because the taxi driver took the long route. He arrived 20 minutes late and apologized.",
        "target": "يوسف تعطل على الونطخوتيان حيت مول الطاكسي دار من الطريق الطويلة. وصل معطل ب20 دقيقة و طلب سماحة.",
    },
    {
        "source": "The old man sat in the café drinking his coffee and reading the newspaper. He does this every morning.",
        "target": "الراجل الكبير جلس فالقهوة كيشرب القهوة ديالو وكيقرا الجورنال. كلا صباح كيدير هاكا.",
    },
    {
        "source": "Khadija and Nadia are best friends. They met in college and have been inseparable ever since.",
        "target": "خديجة ونادية صحابات كي الخوت. تعارفو فالجامعة ومن ديك الساعة ما تفارقوش.",
    },
    {
        "source": "The children played in the park until sunset. Their parents called them when it got dark.",
        "target": "الدراري لعبو فالبارك حتى طاحت الشمس. و والديهم عيطو عليهم ملي ظلام الحال.",
    },
    {
        "source": "Samir found a stray kitten on the street and brought it home. His mother let him keep it.",
        "target": "سمير لقا مش صغير تالف فالزنقة وجابو للدار. ماماه خلاتو يخليه.",
    },

    # ── Questions (common question patterns) ──────────────────────────
    {
        "source": "What's your name?",
        "target": "شنو سميتك؟",
    },
    {
        "source": "Where are you from?",
        "target": "منين نتا؟",
    },
    {
        "source": "How old are you?",
        "target": "شحال فعمرك؟",
    },
    {
        "source": "What do you do for a living?",
        "target": "شنو كتخدم؟",
    },
    {
        "source": "Are you married?",
        "target": "واش نتا مزوج؟",
    },
    {
        "source": "Do you have children?",
        "target": "واش عندك الدراري؟",
    },
    {
        "source": "What's your phone number?",
        "target": "شنو النمرة ديال التيليفون ديالك؟",
    },
    {
        "source": "When is your birthday?",
        "target": "إمتى عيد الميلاد ديالك؟",
    },
    {
        "source": "Where do you live?",
        "target": "فين كتسكن؟",
    },
    {
        "source": "What time does the store open?",
        "target": "وقتاش كيحل الحانوت؟",
    },
    {
        "source": "How far is it?",
        "target": "شحال بعيدة؟",
    },
    {
        "source": "Why are you late?",
        "target": "علاش تعطلتي؟",
    },
    {
        "source": "Who told you that?",
        "target": "شكون قال ليك هادشي؟",
    },
    {
        "source": "Which one do you want?",
        "target": "شمن وحدة بغيتي؟",
    },
    {
        "source": "How did you know?",
        "target": "كيفاش عرفتي؟",
    },

    # ── Commands / imperatives ────────────────────────────────────────
    {
        "source": "Sit down.",
        "target": "جلس.",
    },
    {
        "source": "Stand up.",
        "target": "نوض.",
    },
    {
        "source": "Come here.",
        "target": "أجي هنا.",
    },
    {
        "source": "Go away.",
        "target": "سير فحالك.",
    },
    {
        "source": "Be quiet.",
        "target": "سكت.",
    },
    {
        "source": "Listen to me.",
        "target": "سمع ليا.",
    },
    {
        "source": "Look at this.",
        "target": "شوف هادشي.",
    },
    {
        "source": "Give it to me.",
        "target": "عطيها ليا.",
    },
    {
        "source": "Put it here.",
        "target": "حطها هنا.",
    },
    {
        "source": "Open the door.",
        "target": "حلّ الباب.",
    },
    {
        "source": "Close the window.",
        "target": "سدّ الشرجم.",
    },
    {
        "source": "Stop it.",
        "target": "حبس باراكا.",
    },
    {
        "source": "Don't touch it.",
        "target": "ما تقيسوش.",
    },
    {
        "source": "Leave me alone.",
        "target": "خليني بوحدي.",
    },
    {
        "source": "Help me.",
        "target": "عاوني.",
    },
    {
        "source": "Let's go.",
        "target": "يالله.",
    },
    {
        "source": "Try again.",
        "target": "عاود جرب.",
    },
    {
        "source": "Be careful.",
        "target": "رد بالك.",
    },
    {
        "source": "Don't worry.",
        "target": "ما تخافش.",
    },
    {
        "source": "Calm down.",
        "target": "تكالما.",
    },
    {
        "source": "Pay attention.",
        "target": "ركز.",
    },
    {
        "source": "Think about it.",
        "target": "فكر فيها.",
    },

    # ── Numbers / counting in context ─────────────────────────────────
    {
        "source": "I have two brothers and one sister.",
        "target": "عندي جوج ديال الخوت وخت وحدة.",
    },
    {
        "source": "The school has 500 students.",
        "target": "المدرسة فيها 500 ديال التلاميذ.",
    },
    {
        "source": "He's 25 years old.",
        "target": "عندو 25 عام.",
    },
    {
        "source": "The building has 10 floors.",
        "target": "العمارة فيها 10 ديال الطوابق.",
    },
    {
        "source": "First, second, third.",
        "target": "اللول، الثاني، الثالث.",
    },
    {
        "source": "I need half a kilo of meat.",
        "target": "خاصني نص كيلو ديال اللحم.",
    },
    {
        "source": "Give me a quarter of a kilo of cheese.",
        "target": "عطيني ربع كيلو ديال الفروماج.",
    },

    # ── Sport ─────────────────────────────────────────────────────────
    {
        "source": "What's the score?",
        "target": "شحال النتيجة؟",
    },
    {
        "source": "He scored a goal.",
        "target": "ماركا بيت.",
    },
    {
        "source": "The match starts at 8 PM.",
        "target": "الماتش كيبدا مع 8 ديال لعشية.",
    },
    {
        "source": "Our team won.",
        "target": "الفرقة ديالنا ربحات.",
    },
    {
        "source": "They lost the game.",
        "target": "خسرو الماتش.",
    },
    {
        "source": "He's a good player.",
        "target": "هوا لاعب نادي.",
    },
    {
        "source": "The referee made a bad call.",
        "target": "الحكم دار حكم خايب.",
    },
    {
        "source": "Let's play football.",
        "target": "يالله نلعبو الكورة.",
    },
    {
        "source": "I go to the gym every morning.",
        "target": "كنمشي للصال كلا صباح.",
    },

    # ── Travel ────────────────────────────────────────────────────────
    {
        "source": "I need to book a hotel.",
        "target": "خاصني نشد فشي أوطيل.",
    },
    {
        "source": "Where is the airport?",
        "target": "فين جا المطار؟",
    },
    {
        "source": "I lost my passport.",
        "target": "ضيعت الباسبور ديالي.",
    },
    {
        "source": "The flight is at 6 AM.",
        "target": "الطيارة مع 6 ديال الصباح.",
    },
    {
        "source": "I need a visa.",
        "target": "خاصني الفيزا.",
    },
    {
        "source": "The hotel is fully booked.",
        "target": "الأوطيل عامر.",
    },
    {
        "source": "I want a room with a view.",
        "target": "بغيت شامبر فيها منظر زوين.",
    },
    {
        "source": "What's the check-out time?",
        "target": "فاش كيخصنا نخرجو من الشامبر؟",
    },
    {
        "source": "I forgot my passport at home and went back from the airport empty-handed.",
        "target": "نسيت الباسبور فالدار ورجعت من المطار بيدي خاويين.",
    },
    {
        "source": "The trip from Fez to Tangier takes about 3 hours on the highway.",
        "target": "الطريق من فاس لطنجة كتاخد تقريبا 3 ديال السوايع بلوطوروت.",
    },

    # ── Longer sentences / complex structures ─────────────────────────
    {
        "source": "Last summer, we drove from Marrakech to Essaouira along the coast. The road was beautiful but narrow in some places.",
        "target": "الصيف اللي فات شدينا الطريق من مراكش لصويرة على طول الساحل. الطريق كانت زوينة ولكن ضيقة فشي بلايص.",
    },
    {
        "source": "We stopped twice for tea and arrived just before sunset. The kids ran straight to the beach.",
        "target": "وقفنا جوج مرات باش نشربو أتاي ووصلنا غير قبل الغروب. الدراري جراو ديريكت للبحر.",
    },
    {
        "source": "If I had studied harder, I would have passed the exam.",
        "target": "كون قريت مزيان، كون نجحت فالامتحان.",
    },
    {
        "source": "Even though it was raining, she went outside without an umbrella.",
        "target": "واخا كانت الشتا كتصب، خرجات بلا مضل.",
    },
    {
        "source": "He didn't realize his wallet was missing until he got to the store.",
        "target": "ما عاقش بلي البزطام ديالو ضاع حتى وصل للحانوت.",
    },
    {
        "source": "The more I think about it, the more confused I get.",
        "target": "كل ما فكرت فيها, كل ما زدت دخت كثر.",
    },
    {
        "source": "I wish I had gone to the party, but I was too tired.",
        "target": "مكرهتش كون مشيت للحفلة، ولكن كنت عيان بزاف.",
    },
    {
        "source": "She asked me if I could help her move the furniture.",
        "target": "سولاتني واش نقدر نعاونها ترحل الفراش.",
    },
    {
        "source": "By the time I arrived, everyone had already left.",
        "target": "فاس وصلت، لقيت كلشي ديجا مشا.",
    },
    {
        "source": "He used to work here before he moved to another city.",
        "target": "كان كيخدم هنا قبل ما يرحل لمدينة خرى.",
    },

    # ── Darija→English critical pairs (reverse direction) ─────────────
    {
        "source": "I went to the store and found it closed.",
        "target": "مشيت للحانوت ولقيتو مسدود.",
    },
    {
        "source": "My son refuses to eat vegetables.",
        "target": "ولدي ما بغاش ياكل الخضرة.",
    },
    {
        "source": "The neighbors had a big wedding yesterday. The music lasted until midnight.",
        "target": "الجيران دارو عرس كبير البارح. الموسيقى بقات حتى لنص الليل.",
    },
    {
        "source": "My brother sent me a message saying he'll visit us this weekend.",
        "target": "خويا صيفط ليا ميساج وقال ليا بلي غادي يجي يزورنا هاد الويكاند.",
    },
    {
        "source": "Strike while the iron is hot before the opportunity is gone.",
        "target": "ضرب الحديد وهو سخون قبل ما تضيع الفرصة.",
    },
    {
        "source": "This problem has gotten too big, we don't know where to start anymore.",
        "target": "هاد المشكل كبر بزاف، ما بقيناش عارفين منين نبداو.",
    },
    {
        "source": "He's playing with fire and not listening to anyone.",
        "target": "كيلعب بالعافية وما كيسمعش لحتى واحد.",
    },
    {
        "source": "Fatima Zahra passed the baccalaureate with 18.5 out of 20. Her parents were overjoyed and threw a big dinner.",
        "target": "فاطمة الزهراء نجحات فالباك بميزة 18.5 على 20. الوالدين ديالها فرحو بيها بزاف ودارو ليها عشا كبير.",
    },
    {
        "source": "Said forgot his passport at home and came back from the airport empty-handed.",
        "target": "سعيد نسى الباسبور ديالو فالدار ورجع من المطار خاوي.",
    },

    # ── Conversation fillers / discourse markers ─────────────────────
    {
        "source": "Actually, I changed my mind.",
        "target": "فالحقيقة، بدلت رأي ديالي.",
    },
    {
        "source": "By the way, did you see the news?",
        "target": "اه حجا، واش شفتي الأخبار؟",
    },
    {
        "source": "In my opinion, this is the best option.",
        "target": "بالنسبة ليا, هادا هوا أحسن خيار.",
    },
    {
        "source": "To be honest, I don't like it.",
        "target": "بالصح، ما عجبتنيش.",
    },
    {
        "source": "Anyway, let's continue.",
        "target": "على أي حال، يالله نكملو.",
    },
    {
        "source": "For example, look at this.",
        "target": "مثلاً، شوف هادشي.",
    },
    {
        "source": "In other words, he failed.",
        "target": "يعني، فشل.",
    },
    {
        "source": "Nevertheless, we should try.",
        "target": "ولكن واخا هكاك خاصنا نجربو.",
    },
    {
        "source": "As I was saying, the plan is simple.",
        "target": "بحال كي كنت كنقول، البلان ساهل.",
    },
    {
        "source": "First of all, let me explain.",
        "target": "أولاً، خليني نشرح.",
    },

    # ── More daily phrases & expressions ──────────────────────────────
    {
        "source": "I'm stuck in traffic.",
        "target": "راني حابس فالزحام فالطريق.",
    },
    {
        "source": "Traffic is terrible today.",
        "target": "الطريق عامرة اليوم.",
    },
    {
        "source": "There's a traffic jam.",
        "target": "كاين زحام فالطريق.",
    },
    {
        "source": "I overslept.",
        "target": "داني النعاس.",
    },
    {
        "source": "I couldn't sleep last night.",
        "target": "ما قدرتش ننعس البارح بالليل.",
    },
    {
        "source": "I had a nightmare.",
        "target": "حلمت حلمة خايبة.",
    },
    {
        "source": "I had a nice dream.",
        "target": "حلمت حلمة زوينة.",
    },
    {
        "source": "I'm not in the mood.",
        "target": "ما عنديش الكانة.",
    },
    {
        "source": "I'm fed up.",
        "target": "انا صافي تقاداو ليا.",
    },
    {
        "source": "I'm full.",
        "target": "انا شبعت.",
    },
    {
        "source": "I'm starving.",
        "target": "فيا الموت ديال الجوع.",
    },
    {
        "source": "I have a headache.",
        "target": "كيضرني راسي.",
    },
    {
        "source": "I need a nap.",
        "target": "خاصني نعس شوية.",
    },
    {
        "source": "I'm running late.",
        "target": "أنا راني معطل.",
    },
    {
        "source": "The alarm didn't go off.",
        "target": "الساعة ما خدمتش.",
    },
    {
        "source": "I'll be right back.",
        "target": "انا غادي نرجع دابا.",
    },
    {
        "source": "Hold on a second.",
        "target": "تسنا شوية.",
    },
    {
        "source": "It's my turn.",
        "target": "وصلات نوبتي.",
    },
    {
        "source": "It's not my fault.",
        "target": "ماشي الديفو ديالي.",
    },
    {
        "source": "I don't care.",
        "target": "ماشي شغلي.",
    },
    {
        "source": "It doesn't matter.",
        "target": "ما كاين باس.",
    },
    {
        "source": "I have no idea.",
        "target": "ما عندي حتى فكرة.",
    },
    {
        "source": "Never mind.",
        "target": "بلاش صافي.",
    },
    {
        "source": "Just kidding.",
        "target": "غير كنضحك.",
    },
    {
        "source": "I'm serious.",
        "target": "بالصح، ما كنضحكش.",
    },
    {
        "source": "I swear.",
        "target": "وحق الله.",
    },
    {
        "source": "I promise.",
        "target": "كنواعدك.",
    },
    {
        "source": "I forgot.",
        "target": "نسيت.",
    },
    {
        "source": "I remember now.",
        "target": "دابا عاد تفكرت.",
    },
    {
        "source": "What a coincidence!",
        "target": "واو علا صدفة!",
    },
    {
        "source": "What a shame!",
        "target": "يا حصرة!",
    },
    {
        "source": "What a relief!",
        "target": "الحمد لله!",
    },
    {
        "source": "It's too late.",
        "target": "فات الفوت.",
    },
    {
        "source": "It's about time.",
        "target": "هذا هوا الوقت.",
    },
    {
        "source": "I doubt it.",
        "target": "كنشك فهادشي.",
    },
    {
        "source": "I agree.",
        "target": "متافق.",
    },
    {
        "source": "I disagree.",
        "target": "ما متافقش.",
    },
    {
        "source": "That makes sense.",
        "target": "هادشي معقول.",
    },
    {
        "source": "That's not fair.",
        "target": "هادشي ماشي عدل.",
    },

    # ── Moroccan cities / geography ───────────────────────────────────
    {
        "source": "Casablanca is the biggest city in Morocco.",
        "target": "كازا هيا أكبر مدينة فالمغرب.",
    },
    {
        "source": "Rabat is the capital of Morocco.",
        "target": "الرباط هيا العاصمة ديال المغرب.",
    },
    {
        "source": "Marrakech is known for its souks.",
        "target": "مراكش معروفة بالأسواق ديالها.",
    },
    {
        "source": "Fez has the oldest university in the world.",
        "target": "فاس فيها أقدم جامعة فالعالم.",
    },
    {
        "source": "Tangier is a beautiful port city.",
        "target": "طنجة مدينة ديال الميناء زوينة.",
    },
    {
        "source": "Essaouira is famous for its beaches and wind.",
        "target": "الصويرة معروفة بالشواطئ ديالها والريح.",
    },
    {
        "source": "Chefchaouen is the blue city.",
        "target": "شفشاون هيا المدينة الزرقة.",
    },
    {
        "source": "Agadir has nice weather year-round.",
        "target": "أكادير الجو فيها زوين طول العام.",
    },
    {
        "source": "Meknes is one of the imperial cities.",
        "target": "مكناس وحدة من المدن الإمبراطورية.",
    },
    {
        "source": "Ouarzazate is the gateway to the Sahara.",
        "target": "ورزازات هيا الباب ديال الصحرا.",
    },

    # ── Compound / complex sentences ──────────────────────────────────
    {
        "source": "Even though he was tired, he finished all his work before going home.",
        "target": "واخا كان عيان، كمل الخدمة كاملة قبل ما يمشي للدار.",
    },
    {
        "source": "She not only passed the exam but also got the highest score.",
        "target": "ماشي غير نجحات فالامتحان، ولكن خدات أعلى نقطة.",
    },
    {
        "source": "Unless you study hard, you won't pass.",
        "target": "إلا ما قريتيش مزيان، ما غاديش تنجح.",
    },
    {
        "source": "Whenever I visit my grandmother, she makes me tea.",
        "target": "كل مرة نزور جداتي، كتصاوب ليا أتاي.",
    },
    {
        "source": "The reason he left is that he found a better job.",
        "target": "السبب علاش خرج من الخدمة هوا حيت لقى خدمة حسن.",
    },
    {
        "source": "No matter how hard I try, I can't seem to fix this bug.",
        "target": "واخا شحال حاولت، ما قدرتش نصلح هاد المشكل.",
    },
    {
        "source": "As soon as she heard the news, she started crying.",
        "target": "غير سمعات الخبر، بدات كتبكي.",
    },
    {
        "source": "While he was cooking, the phone rang.",
        "target": "فالوقت اللي كان كيطيب، صونا التيليفون.",
    },
    {
        "source": "Not only is the food delicious, but the service is also excellent.",
        "target": "ماشي غير الماكلة بنينة، ولكن حتى السيرفيس زوين.",
    },
    {
        "source": "The harder you work, the more you learn.",
        "target": "كل ما خدمتي كثر، كل ما تعلمتي كثر.",
    },
]


def build_generated_manual_fix_examples():
    rows = []

    # exact phrase preservation
    exact_terms = ["OK", "DONE", "END", "PASS", "READY"]
    for term in exact_terms:
        rows.append({
            "source": f"Return exactly \"{term}\". Do not add any other words.",
            "target": f"رجّع ليا بالضبط \"{term}\". ما تزيد حتى كلمة خرى.",
        })
        rows.append({
            "source": term,
            "target": term,
        })

    # placeholders
    placeholder_specs = [
        ("[forbidden_words]", "[ender]", "Is there anything else I can help with?"),
        ("[blocked_terms]", "[closing_phrase]", "Thank you for your time."),
        ("[restricted_words]", "[final_line]", "Please let me know."),
    ]
    for list_name, end_name, exact_phrase in placeholder_specs:
        rows.append({
            "source": f"Do not include keywords {list_name} in the response. End the response with {end_name}. {end_name} is \"{exact_phrase}\"",
            "target": f"ما تدخلش الكلمات المفتاحية {list_name} فالجواب. سالي الجواب ب{end_name}. {end_name} هيا \"{exact_phrase}\"",
        })

    # bullets / exact counts
    for n in [3, 4, 5]:
        rows.append({
            "source": f"Your response should contain exactly {n} bullet points.",
            "target": f"الجواب ديالك خاص يكون فيه بالضبط {n} ديال النقط.",
        })

    # preserve code / variables
    code_specs = [
        "`pip install transformers`",
        "`python train.py --lr 5e-5`",
        "`user_name = 'Max'`",
    ]
    for code in code_specs:
        rows.append({
            "source": f"Do not translate anything inside backticks: {code}",
            "target": f"ما تترجم حتى حاجة داخل backticks: {code}",
        })

    # preserve names / places
    entities = [
        ("Max", "ماكس", "Who is", "شكون هوا"),
        ("Lima", "ليما", "What is", "شنو هيا"),
        ("Peru", "بيرو", "What is", "شنو هيا"),
        ("Big Mac", "Big Mac", "What is", "شنو هوا"),
        ("Aztecs", "الأزتيك", "Who are", "شكون هما"),
    ]
    for en, dr, en_q, dr_q in entities:
        rows.append({
            "source": f"Do not change the name {en}.",
            "target": f"ما تبدلش السمية {en}.",
        })
        rows.append({
            "source": f"{en_q} {en}?",
            "target": f"{dr_q} {dr}؟",
        })

    return rows


MANUAL_FIXES.extend(build_generated_manual_fix_examples())

# ============================================================
# Manual eval set (not used for training)
# ============================================================

MANUAL_EVAL = [
    {
        "source": "You love cats and pizza. How excited are you that a new pizza truck in town also serves treats for cats?",
        "target": "نتا كتعشق القطوط والبيتزا. شحال فرحان حيث كاين كاميو جديد ديال البيتزا فالمدينة وكيدير حتى ماكلة للقطوط؟",
    },
    {
        "source": "Finish your response with this exact phrase: \"I can help with that.\"",
        "target": "سالي الجواب ديالك بهاد العبارة بالضبط: \"I can help with that.\"",
    },
    {
        "source": "Do not translate anything inside square brackets: [user_name]",
        "target": "ما تترجم حتى حاجة داخل المعقوفات: [user_name]",
    },
    {
        "source": "Write exactly 4 bullet points and keep everything in lowercase.",
        "target": "كتب بالضبط 4 ديال النقط وخلي كلشي بحروف صغار.",
    },
    {
        "source": "The capital of Japan is Tokyo.",
        "target": "عاصمة اليابان هيا طوكيو.",
    },
    {
        "source": "My booking code is A7P9K.",
        "target": "الكود ديال الحجز ديالي هوا A7P9K.",
    },
    {
        "source": "Let's brainstorm next Thursday at 10:00 AM.",
        "target": "يالله نفكرو فشي أفكار نهار الخميس الجاي مع 10:00 دالصباح.",
    },
    {
        "source": "This is going to be amazing!",
        "target": "هادشي غادي يكون زوين بزاف!",
    },
    # --- round 2 eval: idioms ---
    {
        "source": "He was barking up the wrong tree.",
        "target": "كان غالط.",
    },
    # --- round 2 eval: location preservation ---
    {
        "source": "The headquarters is in Seattle, Washington.",
        "target": "الشركة الرئيسية فـ Seattle، واشنطن.",
    },
    # --- round 2 eval: number / unit preservation ---
    {
        "source": "The item costs $29.99 and weighs 1.2 kg.",
        "target": "السلعة ثمنها $29.99 وكتوزن 1.2 كيلو.",
    },
    # --- round 2 eval: code block preservation ---
    {
        "source": "Run `npm run build` to compile the project.",
        "target": "دير `npm run build` باش تكومبيلي المشروع.",
    },
    # --- round 2 eval: error message preservation ---
    {
        "source": "The server returned 503 Service Unavailable.",
        "target": "السيرفر رجع 503 Service Unavailable.",
    },
    # --- round 2 eval: double negation ---
    {
        "source": "He is completely unqualified for the job.",
        "target": "هوا معندوش فيدو والو لهاد البوسط.",
    },
    # --- round 2 eval: gender tracking ---
    {
        "source": "She asked her mother for advice. Her mother told her to be patient.",
        "target": "طلبات من ماماها تنصحها. ماماها قالت ليها تصبر.",
    },
    # --- round 2 eval: 3rd person narrative ---
    {
        "source": "Omar missed the bus. He had to walk to school. His teacher asked why he was late.",
        "target": "عمر فاتو الطوبيس. مشى للمدرسة على رجليه. الأستاذ سولو علاش تعطل.",
    },
    # --- round 2 eval: hashtag preservation ---
    {
        "source": "Post it with the hashtag #DevMorocco.",
        "target": "حط البوسط بالهاشتاڭ #DevMorocco.",
    },
    # --- round 2 eval: traffic ---
    {
        "source": "The traffic was terrible this morning.",
        "target": "الزحام كان خايب هاد الصباح.",
    },
    # --- round 3 eval: greetings / farewells ---
    {
        "source": "Hi, good to see you!",
        "target": "سلام، نهار كبير هادا من شفتك!",
    },
    {
        "source": "Goodbye and take care.",
        "target": "بسلامة ورد بالك.",
    },
    # --- round 3 eval: bored vs worried ---
    {
        "source": "I'm bored, let's do something fun.",
        "target": "قنطت، يالله نبدلو الجو.",
    },
    # --- round 3 eval: family terms ---
    {
        "source": "My grandmother lives in Fes.",
        "target": "جداتي ساكنة ففاس.",
    },
    # --- round 3 eval: taxi ---
    {
        "source": "The taxi driver got lost.",
        "target": "مول الطاكسي تلف.",
    },
    # --- round 3 eval: call me DR→EN ---
    {
        "source": "She called me three times.",
        "target": "عيطات ليا ثلث مرات.",
    },
    # --- round 3 eval: idiom ---
    {
        "source": "He's pulling your leg.",
        "target": "كيضحك عليك.",
    },
    {
        "source": "Don't put all your eggs in one basket.",
        "target": "متحطش كلشي فبلاصة وحدة.",
    },
    # --- round 3 eval: gender tracking ---
    {
        "source": "She graduated top of her class. Her parents were proud.",
        "target": "تخرجات هيا الأولى فالقسم ديالها. والديها كانو فخورين بيها.",
    },
    # --- round 3 eval: explicit gender ---
    {
        "source": "He is my teacher. She is my doctor.",
        "target": "هوا أستاذ ديالي. هيا طبيبة ديالي.",
    },
    # --- round 3 eval: daily life ---
    {
        "source": "I'm tired and hungry.",
        "target": "عييت وفيا الجوع.",
    },
    # --- round 3 eval: French loanwords ---
    {
        "source": "I missed the bus this morning.",
        "target": "فاتني الطوبيس هاد الصباح.",
    },
    # --- round 3 eval: Darija idiom DR→EN ---
    {
        "source": "He acted like it was none of his business.",
        "target": "دار كيبحال الى ماشي شغلو.",
    },
]

# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# Load and split base dataset
# ============================================================

base_ds = load_dataset(DATASET_ID, split="train")
print("Base rows:", len(base_ds))
print("Columns:", base_ds.column_names)
print("Example:", base_ds[0])

split_ds = base_ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_base = split_ds["train"]
val_base = split_ds["test"]

print("Train base:", len(train_base))
print("Val base:", len(val_base))

# ============================================================
# Build prompt-completion rows
# ============================================================


def build_forward_row(source, target):
    return {
        "prompt": [
            {"role": "system", "content": SYS_EN_DR},
            {"role": "user", "content": source},
        ],
        "completion": [
            {"role": "assistant", "content": target},
        ],
    }


def build_reverse_row(source, target):
    return {
        "prompt": [
            {"role": "system", "content": SYS_DR_EN},
            {"role": "user", "content": target},
        ],
        "completion": [
            {"role": "assistant", "content": source},
        ],
    }


train_rows = []
val_rows = []

# --- KEEP DATASET LOADED, BUT DON'T USE IT FOR TRAINING ---
# We are currently bypassing `train_base` to strictly train on manual corrections.
# for ex in train_base:
#     source = (ex["source"] or "").strip()
#     target = (ex["target"] or "").strip()
#     if source and target:
#         train_rows.append(build_forward_row(source, target))

print("Skipping base dataset training rows. Using ONLY manual fixes for training.")

# manual fixes: all forward
for ex in MANUAL_FIXES:
    source = ex["source"].strip()
    target = ex["target"].strip()
    train_rows.append(build_forward_row(source, target))

# manual fixes: 20% reverse (Bidirectional)
manual_reverse_indices = set(
    random.sample(range(len(MANUAL_FIXES)), int(
        len(MANUAL_FIXES) * BIDIRECTIONAL_FRACTION))
)

for i, ex in enumerate(MANUAL_FIXES):
    if i in manual_reverse_indices:
        source = ex["source"].strip()
        target = ex["target"].strip()
        train_rows.append(build_reverse_row(source, target))

# validation: use MANUAL_EVAL (small, curated, unseen during training)
for ex in MANUAL_EVAL:
    source = ex["source"].strip()
    target = ex["target"].strip()
    val_rows.append(build_forward_row(source, target))

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

print(f"\nForward manual fixes: {len(MANUAL_FIXES):,}")
print(f"Reverse manual fixes: {len(manual_reverse_indices):,}")
print(f"Total train rows:     {len(train_ds):,}")
print(f"Validation rows:      {len(val_ds):,}")

# save manual eval for later testing
with open("manual_eval_translation.jsonl", "w", encoding="utf-8") as f:
    for ex in MANUAL_EVAL:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Saved manual eval set to manual_eval_translation.jsonl")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/503 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/592 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/59679 [00:00<?, ? examples/s]

Base rows: 59679
Columns: ['row_id', 'turn_id', 'role', 'source', 'target']
Example: {'row_id': 0, 'turn_id': 0, 'role': 'user', 'source': 'What is the capital of Peru? Your response should contain at least 3 sentences.', 'target': 'شنو هي عاصمة بيرو؟ الجواب ديالك خاصو يكون فيه على الأقل 3 جمل.'}
Train base: 59082
Val base: 597
Skipping base dataset training rows. Using ONLY manual fixes for training.

Forward manual fixes: 774
Reverse manual fixes: 154
Total train rows:     928
Validation rows:      31
Saved manual eval set to manual_eval_translation.jsonl


In [ ]:
# ============================================================
# QLoRA 4-bit
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    #device_map={"": 0},
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# ============================================================
# LoRA
# ============================================================

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/148 [00:00<?, ?B/s]

In [ ]:
# ============================================================
# Training config
# ============================================================

training_args = SFTConfig(
    output_dir="tiny-aya-darija-v3-v4",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    report_to="none",
    seed=SEED,
    max_length=1024,
    packing=False,
)

# ============================================================
# Trainer
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"\n{'='*70}")
print("Translation SFT")
print(f"Base model:     {MODEL_ID}")
print(f"Dataset:        {DATASET_ID}")
print(f"Val base:       {len(val_base):,}")
print(f"Manual fixes:   {len(MANUAL_FIXES):,}")
print(f"Reverse fixes:  {len(manual_reverse_indices):,}")
print(f"Total train:    {len(train_ds):,}")
print(f"Total val:      {len(val_ds):,}")
print(f"Epochs:         {training_args.num_train_epochs}")
print(f"LoRA r:         {peft_config.r}")
print(f"Max length:     {training_args.max_length}")
print(f"LR:             {training_args.learning_rate}")
print(
    f"Batch:          {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps}")
print(f"{'='*70}\n")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/928 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/928 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/31 [00:00<?, ? examples/s]


Translation SFT
Base model:     Lyte/tiny-aya-darija-v3
Dataset:        Lyte/moroccan-instruct-59k
Val base:       597
Manual fixes:   774
Reverse fixes:  154
Total train:    928
Total val:      31
Epochs:         3
LoRA r:         16
Max length:     1024
LR:             0.0001
Batch:          4 x 4



In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6}.


Epoch,Training Loss,Validation Loss
1,0.817544,1.126896
2,0.553899,1.048686
3,0.306114,1.075362


TrainOutput(global_step=174, training_loss=0.6616940169498838, metrics={'train_runtime': 3888.7073, 'train_samples_per_second': 0.716, 'train_steps_per_second': 0.045, 'total_flos': 5301739558649856.0, 'train_loss': 0.6616940169498838})

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

BASE_MODEL = "Lyte/tiny-aya-darija-v3"
ADAPTER_PATH = "tiny-aya-darija-v3-v4/checkpoint-174"  # best checkpoint (epoch 2)
HF_REPO = "Lyte/tiny-aya-darija-v4"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
#tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Load LoRA adapter on top
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Merge and save
print("Merging LoRA adapter...")
merged_model = model.merge_and_unload()

print("Saving locally...")
merged_model.save_pretrained("darija-v4-merged", safe_serialization=True)
tokenizer.save_pretrained("darija-v4-merged")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Merging LoRA adapter...
Saving locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('darija-v4-merged/tokenizer_config.json',
 'darija-v4-merged/chat_template.jinja',
 'darija-v4-merged/tokenizer.json')

In [ ]:
# ── Merge & Push ──────────────────────────────────────────────────────────────

HF_REPO = "Lyte/tiny-aya-darija-v4"

#print("\nMerging LoRA adapter...")
#merged_model = model.merge_and_unload()

#print(f"Saving locally...")
#merged_model.save_pretrained("darija-v4-merged", safe_serialization=True)
#tokenizer.save_pretrained("darija-v4-merged")

print(f"Pushing to {HF_REPO}...")
merged_model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

print(f"\n✅ Pushed to https://huggingface.co/{HF_REPO}")

Pushing to Lyte/tiny-aya-darija-v4...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Pushed to https://huggingface.co/Lyte/tiny-aya-darija-v4


In [ ]:
SYS_EN_DR = (
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the Darija translation."
)

SYS_DR_EN = (
    "Translate the following Moroccan Darija (الدارجة المغربية) text into English. "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the English translation."
)

def infer(system_prompt, text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to(model.device)

    outputs = model.generate(
        input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.3,
        top_p=0.95,
        #top_k=300,
        min_p=0.05,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    gen_text = tokenizer.decode(
        outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return gen_text.strip()

CONV_EDGE_TESTS = [
    # 1. Long rambling story with filler words
    "So basically, I was at the mall yesterday, right, and I bumped into my old high school friend that "
    "I haven't seen in like 7 years. And she was like 'oh my God is that you?' and I was like 'no way!' "
    "and we hugged and started catching up and she told me she got married, had two kids, moved to "
    "Marrakech, opened a bakery, and honestly I felt like I haven't done anything with my life compared to her.",

    # 2. Emotional vent with contradictions
    "I don't even know why I'm upset honestly. Like, he didn't do anything wrong, but at the same time "
    "he didn't do anything right either. He just... exists. And that's the problem. I want someone who "
    "actually tries, you know? Not someone who just shows up and thinks that's enough. Maybe I'm asking "
    "for too much. Or maybe I'm not asking for enough. I don't know anymore.",

    # 3. Giving complex directions with landmarks
    "Okay so to get to my house, you take the highway toward Casablanca, exit at the second roundabout "
    "after the Shell gas station, then go straight for about 500 meters until you see a mosque on your "
    "left. Turn right just after the mosque, then take the first left. My building is the tall white one "
    "with a pharmacy on the ground floor. Apartment 4B, third floor. Ring the bell twice.",

    # 4. Heated argument style
    "You know what your problem is? You never listen. I told you a hundred times not to lend money to "
    "Khalid because he never pays back, and what did you do? You lent him 3000 dirhams! And now he's "
    "not answering your calls. Surprise, surprise. Don't come crying to me about it. I warned you. "
    "I literally warned you. But no, you always think you know better than everyone else.",

    # 5. Planning a trip with friends (group chat style)
    "Alright guys here's the plan for the weekend. We leave Friday after work, should be around 6pm. "
    "Youssef is driving, so everyone pitch in for gas. We're stopping in Beni Mellal for dinner, "
    "I know a great spot that does amazing grilled meat. Then we continue to Ouzoud and camp near "
    "the waterfalls. Saturday we hike, swim, take pictures. Sunday we head back early because Amine "
    "has work at 2pm. Everyone bring a sleeping bag and at least 100 dirhams for food. Any questions?",

    # 6. Tech support frustration
    "I've been trying to fix this laptop for three days now. First the screen started flickering, "
    "then the keyboard stopped working, and now it won't even turn on. I took it to the repair shop "
    "near Derb Ghallef and the guy said the motherboard is fried and it'll cost 2500 dirhams to fix. "
    "That's almost the price of a new laptop! I bought this thing 6 months ago. Honestly, I should "
    "have just bought the MacBook like my brother told me to.",

    # 7. Long cooking explanation
    "The secret to a perfect rfissa is patience. First, you make the msemen the night before and "
    "let them dry out a little. The lentils need to be cooked separately until they're almost mushy. "
    "The chicken has to be slow-cooked with ras el hanout, saffron, onions, and fenugreek for at "
    "least two hours until the meat falls off the bone. Then you tear the msemen into pieces, layer "
    "them in a big plate, pour the lentils over, put the chicken on top, and drizzle the sauce over "
    "everything. My mother adds a handful of fresh coriander at the end and honestly that's what "
    "makes the difference.",

    # 8. Sarcastic response to a humble brag
    "Oh you went to Dubai for the third time this year? That's so amazing. Meanwhile, I can barely "
    "afford to take the bus to work. But sure, tell me more about your infinity pool and your brunch "
    "at that fancy restaurant. I'm really happy for you. Really. No, I'm not jealous at all. Why would "
    "I be jealous? It's not like I've been saving for six months just to fix my car's AC.",

    # 9. Gossipy multi-person narrative
    "Girl, you won't believe what happened at the wedding last night. So Fatima showed up wearing the "
    "EXACT same dress as the bride's sister. And the sister was so mad she didn't even say hello to "
    "anyone the whole night. Then Rachid got into a fight with his cousin over a parking spot outside "
    "the venue. The DJ played the wrong song for the couple's first dance. And to top it all off, "
    "the cake fell over when they were cutting it. Honestly it was the most chaotic wedding I've ever "
    "been to, but the food was incredible.",

    # 10. Philosophical ramble
    "Sometimes I wonder if we're all just pretending to know what we're doing. Like, everyone walks "
    "around acting so confident and sure of themselves, but deep down we're all just figuring it out "
    "as we go. My father is 60 years old and he told me last week that he still doesn't feel like an "
    "adult. That scared me and comforted me at the same time. Maybe life isn't about having all the "
    "answers. Maybe it's about being okay with not knowing.",

    # 11. Mixed technical + casual (explaining to a non-tech friend)
    "Okay so imagine the internet is like a highway, right? And your WiFi router is like the on-ramp. "
    "When too many devices connect at once — your phone, my phone, the TV, the PlayStation, your "
    "sister's laptop — it's like a traffic jam on the highway and everything slows down. That's why "
    "your YouTube keeps buffering. You need to either disconnect some devices or call Maroc Telecom "
    "and upgrade to fiber. The 100 Mbps plan is like 400 dirhams a month I think.",

    # 12. Negotiating at a souk
    "Brother, 800 dirhams for this bag? Come on, you're joking. I can get the same one in Derb Ghallef "
    "for 200. Look, I'll give you 350 and that's my final offer. No? Okay fine, 400, but you throw in "
    "that leather wallet too. Deal? You're killing me here. Fine, 450 for both, but only because my "
    "wife likes the color. Wrap them up.",
]

print("\n" + "="*50)
print("TESTING: ENGLISH -> MOROCCAN DARIJA")
print("="*50)
for text in CONV_EDGE_TESTS:
    print(f"\n[EN] {text}")
    print(f"[DR] {infer(SYS_EN_DR, text)}")


TESTING: ENGLISH -> MOROCCAN DARIJA

[EN] So basically, I was at the mall yesterday, right, and I bumped into my old high school friend that I haven't seen in like 7 years. And she was like 'oh my God is that you?' and I was like 'no way!' and we hugged and started catching up and she told me she got married, had two kids, moved to Marrakech, opened a bakery, and honestly I felt like I haven't done anything with my life compared to her.
[DR] فالأصل، كنت فالسيرفيس البار البارح، بصح وطاحت على صاحبي القديم من المدرسة الثانوية اللي ما شفتوش بحال 7 سنين. وقال ليا 'Oh my God is that you?'. وكنت كنقول 'No way!'. وحنا تبوسينا وبدينا كنهضرو وكتعاود ليا بلي تزوجات وعندها جوج وليدات وانتقلات لمراكش ودارت مخبزة. بصح حسيت بلي ما درتش حتى شي حاجة مقارنة مع حياتها.

[EN] I don't even know why I'm upset honestly. Like, he didn't do anything wrong, but at the same time he didn't do anything right either. He just... exists. And that's the problem. I want someone who actually tries, you know? Not someon

In [ ]:
# Sampling Off

# ======================================================================
# UNSEEN TEST CASES
# ======================================================================
tests_en_to_dr = [
    # --- Strict instruction & Formatting ---
    "Your response should be exactly 2 sentences. Use the keywords: [budget] and [project]. Do not use any technical jargon.",
    "Respond with exactly 'Confirmed'. Nothing else. Then add the hashtag #Success.",
    "Output the result as a markdown list starting with a hyphen -. Do not wrap the output in quotes.",
    "Start your answer with 'Hello' and end it with [DONE]. Keep everything in lowercase.",

    # --- Contextual idioms & descriptive text ---
    "He was completely over the moon when he got his new puppy. They are like two peas in a pod.",
    "She hit the ground running with her new job and knocked it out of the park on day one.",
    "I'm feeling a bit under the weather today, so I think I'll just stay in bed.",
    "Don't worry about it, it's just water under the bridge now. Let's move on.",
    "He told me a white lie so he wouldn't let the cat out of the bag.",

    # --- 3rd person narrative tracking & Gender ---
    "Youssef woke up late today because his alarm didn't ring. He quickly drank his coffee, grabbed his backpack, and ran to catch the train to the university.",
    "Amina is planning a surprise for her sister's graduation. She already ordered the cake and invited all the family members without letting her sister know.",
    "The manager fired his best employee because she was constantly showing up late to meetings.",
    "When Samir saw his father, he ran up to him and gave him a big hug.",

    # --- Entity preservation (Places, Values, Names) ---
    "The startup is located in Toronto, Canada, and they just raised $2.5 million in funding last Monday.",
    "I want to order a Double Cheeseburger, medium fries, and a Coke. My order number is #9099.",
    "They went on a vacation to Tokyo, Japan and bought the new PlayStation 5 Pro.",
    "My email is test.user_99@gmail.com and the temporary pin is Xk-9G-2b.",
    "According to the CEO Elon Musk, Tesla is planning a new factory in Austin, Texas.",

    # --- Technical & Code preservation ---
    "The API returned a 403 Forbidden error because the auth token: `eyJhbGciOi` expired.",
    "Run the following command in your terminal: `npm run dev -- --port 3000`",
    "Change the python dictionary to `{'status': True, 'count': 0}` and save the file.",
    "The stack trace shows: Uncaught ReferenceError: backend_url is not defined.",

    # --- Negative constraints (Don't hallucinate) ---
    "She did not say that the project was cancelled, she just said it was delayed.",
    "It's not that I don't want to go to the party, I'm just incredibly exhausted from work.",
    "I was not entirely convinced by his argument, but I didn't want to interrupt him.",

    # --- Daily Moroccan situations (Testing natural vocabulary mapping) ---
    "I need to go to the grocery store to buy tomatoes, onions, olive oil, and some fresh mint.",
    "There was a massive traffic jam on the highway, we were stuck in the car for almost three hours.",
    "My Wi-Fi is super slow today, I can't even download a simple PDF file.",
]

tests_dr_to_en = [
    # --- Core idiom translations ---
    "الزحام كان خايب بزاف فكازا هاد الصباح. بقيت حاصل فالفوطوي ديالي فالطونوبيل شي ساعتين.",
    "جابها لاصقة فاش قال بلي خصنا نخدمو مجموعين باش ننجحو فالمشروع للي غادي.",
    "كتصب الشتا خيط من السما، ما نقدرش نخرج دابا بمرة.",
    "ديك البنت طارت ليها الفرحة ملي نجحات فالامتحان ديالة الباك.",

    # --- Formatting constraints ---
    "سالي الجواب ديالك بهاد العبارة بالضبط: [END]. ما تزيد حتى كلمة خرى.",
    "عطيني 3 ديال النقط على شكل ليستة مرقمة وبلا ما تشرح والو.",

    # --- Values, tech, API ---
    "الخطاء لي طرا هوا: 500 Internal Server Error. عاود جرب من بعد.",
    "دخلو بـ `npm install react-router-dom` فالتيرمينال.",
    "الثمن هو $15.99، ماشي €15.99. خلص بالكارط ديالك.",

    # --- Everyday conversation ---
    "نسيت الساروت ديال الدار فالخدمة، دابا خصني نتسنى خويا يجي يحل عليا.",
    "الماكلة ديال هاد الريسطو زوينة بزاف، كيحمقني الطاجين ديال اللحم بالبرقوق دياولهم.",
    "الطيارة غادية تطير مع 10:30 دالصباح من البوابة C4.",
]

print("\n" + "="*50)
print("TESTING: ENGLISH -> MOROCCAN DARIJA")
print("="*50)
for text in tests_en_to_dr:
    print(f"\n[EN] {text}")
    print(f"[DR] {infer(SYS_EN_DR, text)}")

print("\n" + "="*50)
print("TESTING: MOROCCAN DARIJA -> ENGLISH")
print("="*50)
for text in tests_dr_to_en:
    print(f"\n[DR] {text}")
    print(f"[EN] {infer(SYS_DR_EN, text)}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



TESTING: ENGLISH -> MOROCCAN DARIJA

[EN] Your response should be exactly 2 sentences. Use the keywords: [budget] and [project]. Do not use any technical jargon.
[DR] الجواب ديالك خاص يكون بالضبط 2 ديال الجمل. استعمل الكلمات المفتاحية: [budget] و [project]. ما تستعملش شي لغة تقنية.

[EN] Respond with exactly 'Confirmed'. Nothing else. Then add the hashtag #Success.
[DR] جاوب غير بـ 'مؤكد'. ما تزيد حتى حاجة خرى. من بعد زيد الهاشتاڭ #Success.

[EN] Output the result as a markdown list starting with a hyphen -. Do not wrap the output in quotes.
[DR] خرج النتيجة على شكل ليستة ديال الماركات باديا ب- . ما تدخلش النتائج فالاقتباس.

[EN] Start your answer with 'Hello' and end it with [DONE]. Keep everything in lowercase.
[DR] بدا الجواب ديالك ب 'سلام' وسلّي ب [DONE]. خلي كلشي lowercase.

[EN] He was completely over the moon when he got his new puppy. They are like two peas in a pod.
[DR] كان فرحان بزاف ملي جاب الكلب الجديد ديالو. هوما بحال جوج ديال البريوات.

[EN] She hit the ground running w

In [ ]:
#max_new_tokens=256,do_sample=True,temperature=0.3,top_p=0.98,top_k=300,

SYS_EN_DR = (
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the Darija translation."
)

SYS_DR_EN = (
    "Translate the following Moroccan Darija (الدارجة المغربية) text into English. "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the English translation."
)

def infer(system_prompt, text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to(model.device)

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.98,
        top_k=300,
        pad_token_id=tokenizer.eos_token_id
    )

    gen_text = tokenizer.decode(
        outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return gen_text.strip()


# ======================================================================
# UNSEEN TEST CASES
# ======================================================================
tests_en_to_dr = [
    # --- Strict instruction & Formatting ---
    "Your response should be exactly 2 sentences. Use the keywords: [budget] and [project]. Do not use any technical jargon.",
    "Respond with exactly 'Confirmed'. Nothing else. Then add the hashtag #Success.",
    "Output the result as a markdown list starting with a hyphen -. Do not wrap the output in quotes.",
    "Start your answer with 'Hello' and end it with [DONE]. Keep everything in lowercase.",

    # --- Contextual idioms & descriptive text ---
    "He was completely over the moon when he got his new puppy. They are like two peas in a pod.",
    "She hit the ground running with her new job and knocked it out of the park on day one.",
    "I'm feeling a bit under the weather today, so I think I'll just stay in bed.",
    "Don't worry about it, it's just water under the bridge now. Let's move on.",
    "He told me a white lie so he wouldn't let the cat out of the bag.",

    # --- 3rd person narrative tracking & Gender ---
    "Youssef woke up late today because his alarm didn't ring. He quickly drank his coffee, grabbed his backpack, and ran to catch the train to the university.",
    "Amina is planning a surprise for her sister's graduation. She already ordered the cake and invited all the family members without letting her sister know.",
    "The manager fired his best employee because she was constantly showing up late to meetings.",
    "When Samir saw his father, he ran up to him and gave him a big hug.",

    # --- Entity preservation (Places, Values, Names) ---
    "The startup is located in Toronto, Canada, and they just raised $2.5 million in funding last Monday.",
    "I want to order a Double Cheeseburger, medium fries, and a Coke. My order number is #9099.",
    "They went on a vacation to Tokyo, Japan and bought the new PlayStation 5 Pro.",
    "My email is test.user_99@gmail.com and the temporary pin is Xk-9G-2b.",
    "According to the CEO Elon Musk, Tesla is planning a new factory in Austin, Texas.",

    # --- Technical & Code preservation ---
    "The API returned a 403 Forbidden error because the auth token: `eyJhbGciOi` expired.",
    "Run the following command in your terminal: `npm run dev -- --port 3000`",
    "Change the python dictionary to `{'status': True, 'count': 0}` and save the file.",
    "The stack trace shows: Uncaught ReferenceError: backend_url is not defined.",

    # --- Negative constraints (Don't hallucinate) ---
    "She did not say that the project was cancelled, she just said it was delayed.",
    "It's not that I don't want to go to the party, I'm just incredibly exhausted from work.",
    "I was not entirely convinced by his argument, but I didn't want to interrupt him.",

    # --- Daily Moroccan situations (Testing natural vocabulary mapping) ---
    "I need to go to the grocery store to buy tomatoes, onions, olive oil, and some fresh mint.",
    "There was a massive traffic jam on the highway, we were stuck in the car for almost three hours.",
    "My Wi-Fi is super slow today, I can't even download a simple PDF file.",
]

tests_dr_to_en = [
    # --- Core idiom translations ---
    "الزحام كان خايب بزاف فكازا هاد الصباح. بقيت حاصل فالفوطوي ديالي فالطونوبيل شي ساعتين.",
    "جابها لاصقة فاش قال بلي خصنا نخدمو مجموعين باش ننجحو فالمشروع للي غادي.",
    "كتصب الشتا خيط من السما، ما نقدرش نخرج دابا بمرة.",
    "ديك البنت طارت ليها الفرحة ملي نجحات فالامتحان ديالة الباك.",

    # --- Formatting constraints ---
    "سالي الجواب ديالك بهاد العبارة بالضبط: [END]. ما تزيد حتى كلمة خرى.",
    "عطيني 3 ديال النقط على شكل ليستة مرقمة وبلا ما تشرح والو.",

    # --- Values, tech, API ---
    "الخطاء لي طرا هوا: 500 Internal Server Error. عاود جرب من بعد.",
    "دخلو بـ `npm install react-router-dom` فالتيرمينال.",
    "الثمن هو $15.99، ماشي €15.99. خلص بالكارط ديالك.",

    # --- Everyday conversation ---
    "نسيت الساروت ديال الدار فالخدمة، دابا خصني نتسنى خويا يجي يحل عليا.",
    "الماكلة ديال هاد الريسطو زوينة بزاف، كيحمقني الطاجين ديال اللحم بالبرقوق دياولهم.",
    "الطيارة غادية تطير مع 10:30 دالصباح من البوابة C4.",
]

print("\n" + "="*50)
print("TESTING: ENGLISH -> MOROCCAN DARIJA")
print("="*50)
for text in tests_en_to_dr:
    print(f"\n[EN] {text}")
    print(f"[DR] {infer(SYS_EN_DR, text)}")

print("\n" + "="*50)
print("TESTING: MOROCCAN DARIJA -> ENGLISH")
print("="*50)
for text in tests_dr_to_en:
    print(f"\n[DR] {text}")
    print(f"[EN] {infer(SYS_DR_EN, text)}")


TESTING: ENGLISH -> MOROCCAN DARIJA

[EN] Your response should be exactly 2 sentences. Use the keywords: [budget] and [project]. Do not use any technical jargon.
[DR] الجواب ديالك خاص يكون بالضبط 2 ديال الجمل. استعمل الكلمات المفتاحية: [budget] و [project]. ما تستعملش شي لغة تقنية.

[EN] Respond with exactly 'Confirmed'. Nothing else. Then add the hashtag #Success.
[DR] جاوب غير بـ 'مؤكد'. ما تزيد حتى حاجة خرى. من بعد زيد الهاشتاڭ #Success.

[EN] Output the result as a markdown list starting with a hyphen -. Do not wrap the output in quotes.
[DR] خرج النتيجة على شكل ليستة ديال الماركات باديا ب- . ما تدخلش النتيجة فالاقتباس.

[EN] Start your answer with 'Hello' and end it with [DONE]. Keep everything in lowercase.
[DR] بدا الجواب ديالك بـ 'سلام' وسلّي ب [DONE]. خلي كلشي lowercase.

[EN] He was completely over the moon when he got his new puppy. They are like two peas in a pod.
[DR] كان فرحان بزاف ملي جاب الكلب الجديد ديالو. هوما بحال جوج ديال البريزات.

[EN] She hit the ground running 

In [ ]:
#max_new_tokens=512, temperature=temp, top_p=0.9, do_sample=True
# ── Comprehensive sanity check ────────────────────────────────────────────────

print(f"\n{'='*70}")
print("  🧪 SANITY CHECK")
print(f"{'='*70}")

model.eval()

def test_translate(text, sys_prompt, temp=0.3):
    msgs = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=temp, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

SYS_ED = "Translate the following English text into Moroccan Darija (الدارجة المغربية). Output ONLY the Darija translation."
SYS_DE = "Translate the following Moroccan Darija (الدارجة المغربية) text into English. Output ONLY the English translation."
SYS_FD = "Translate the following French text into Moroccan Darija. Output ONLY the Darija translation."
SYS_DF = "Translate the following Moroccan Darija text into French. Output ONLY the French translation."
SYS_EF = "Translate the following English text into French. Output ONLY the French translation."

tests = [
    # Short EN→DR (previously broken)
    ("EN→DR", SYS_ED, "I love Morocco."),
    ("EN→DR", SYS_ED, "Hello"),
    ("EN→DR", SYS_ED, "Goodbye"),
    ("EN→DR", SYS_ED, "No thanks."),
    ("EN→DR", SYS_ED, "I'm bored."),
    ("EN→DR", SYS_ED, "Call me later."),

    # Unseen EN→DR
    ("EN→DR*", SYS_ED, "I love Agadir."),
    ("EN→DR*", SYS_ED, "The elevator is broken."),
    ("EN→DR*", SYS_ED, "My cat ran away."),
    ("EN→DR*", SYS_ED, "I need to buy groceries."),

    # Medium
    ("EN→DR", SYS_ED, "My grandmother makes the best tagine in the world."),
    ("EN→DR", SYS_ED, "The taxi driver took the long route and charged me double."),

    # Math
    ("MATH", SYS_ED, "To solve 2x + 5 = 17, subtract 5 to get 2x = 12, divide by 2 to get x = 6."),
    ("MATH", SYS_ED, "Ahmed has 3 boxes with 12 oranges each. He gives away 7 and eats 2. Remaining: 36 - 9 = 27."),
    ("MATH*", SYS_ED, "If a car travels at 80 km/h for 2.5 hours, the distance is 80 × 2.5 = 200 km."),

    # Code
    ("CODE", SYS_ED, "The error 'IndexError: list index out of range' means you're accessing a position that doesn't exist."),
    ("CODE", SYS_ED, "A REST API uses GET, POST, PUT, DELETE methods."),
    ("CODE*", SYS_ED, "A for loop like 'for item in my_list: print(item)' iterates over each element."),

    # Science
    ("SCI", SYS_ED, "A black hole forms when a massive star collapses. Not even light can escape."),
    ("SCI", SYS_ED, "Photosynthesis converts CO₂ and H₂O into glucose and oxygen: 6CO₂ + 6H₂O → C₆H₁₂O₆ + 6O₂."),
    ("SCI*", SYS_ED, "Ohm's law: V = IR. If I = 2A and R = 5Ω, then V = 10V."),

    # DR→EN
    ("DR→EN", SYS_DE, "كنبغي المغرب."),
    ("DR→EN", SYS_DE, "بسلامة"),
    ("DR→EN", SYS_DE, "لا شكراً."),
    ("DR→EN", SYS_DE, "عي�� ليا من بعد."),
    ("DR→EN", SYS_DE, "باش نحلو المعادلة 2x + 5 = 17، أولا نطرحو 5 من الجهتين."),
    ("DR→EN", SYS_DE, "الثقب الأسود كيتكون فاش نجم كبير كينهار تحت الجاذبية ديالو."),

    # FR→DR
    ("FR→DR", SYS_FD, "J'aime le Maroc."),
    ("FR→DR", SYS_FD, "Mon fils a réussi son baccalauréat avec mention."),
    ("FR→DR*", SYS_FD, "La pollution de l'air est un problème majeur dans les grandes villes."),

    # DR→FR
    ("DR→FR", SYS_DF, "كنبغي المغرب."),
    ("DR→FR", SYS_DF, "نسيت الباسبور ديالي فالدار."),
    ("DR→FR*", SYS_DF, "الطاكسي دار الطريق الطويلة وخلصني الضعف."),

    # EN→FR
    ("EN→FR", SYS_EF, "Artificial intelligence is transforming healthcare."),
    ("EN→FR", SYS_EF, "Binary search has a time complexity of O(log n)."),
    ("EN→FR*", SYS_EF, "The patient was diagnosed with a bacterial infection and prescribed antibiotics."),
]

for label, sys_prompt, text in tests:
    r = test_translate(text, sys_prompt)
    print(f"\n  [{label}] {text[:80]}{'...' if len(text)>80 else ''}")
    print(f"  → {r[:150]}{'...' if len(r)>150 else ''}")

# Round-trips
print(f"\n  --- Round-trips ---")
roundtrips = [
    "The Pythagorean theorem states that a² + b² = c² in a right triangle.",
    "A REST API uses GET, POST, PUT, and DELETE methods.",
    "My grandmother makes the best tagine in the world.",
    "To solve 2x + 5 = 17, subtract 5 to get 2x = 12, then divide by 2.",
]
for text in roundtrips:
    dr = test_translate(text, SYS_ED)
    back = test_translate(dr, SYS_DE)
    print(f"\n  🇬🇧 {text}")
    print(f"  🇲🇦 {dr[:120]}")
    print(f"  🇬🇧 {back[:120]}")

print(f"\n{'='*70}")
print("  ✅ DONE! * = unseen during training")
print(f"{'='*70}")


  🧪 SANITY CHECK

  [EN→DR] I love Morocco.
  → كنبغي المغرب.

  [EN→DR] Hello
  → سلام

  [EN→DR] Goodbye
  → بسلامة

  [EN→DR] No thanks.
  → لا شكراً.

  [EN→DR] I'm bored.
  → جاني الملل.

  [EN→DR] Call me later.
  → عيط ليا من بعد.

  [EN→DR*] I love Agadir.
  → كنبغي أكادير.

  [EN→DR*] The elevator is broken.
  → المصعد خاسر.

  [EN→DR*] My cat ran away.
  → القط ديالي هرب.

  [EN→DR*] I need to buy groceries.
  → خاصني نتقضى.

  [EN→DR] My grandmother makes the best tagine in the world.
  → جداتي كتصاوب أحسن طاجين فالعالم.

  [EN→DR] The taxi driver took the long route and charged me double.
  → السائق ديال الطاكسي دار الطريق الطويلة وزاد عليا الدوبل.

  [MATH] To solve 2x + 5 = 17, subtract 5 to get 2x = 12, divide by 2 to get x = 6.
  → باش نحلو 2x + 5 = 17، نقصو 5 باش تعطينا 2x = 12، قسمو على 2 باش تعطينا x = 6.

  [MATH] Ahmed has 3 boxes with 12 oranges each. He gives away 7 and eats 2. Remaining: 3...
  → أحمد عندو 3 ديال الصناديق فيها 12 ديال الليمون فكل صندوق. عطى 7 و

In [ ]:
from datasets import load_dataset
import torch
import re

# ── load a tiny sample to experiment ─────────────────────────────────────────
das = load_dataset("KandirResearch/Short-Instruct-570k", split="train[:20]")

# ── pick device from model safely ────────────────────────────────────────────
device = next(model.parameters()).device

# ── peek at the structure ────────────────────────────────────────────────────
sample = das[0]["messages"]
for turn in sample:
    print(f"[{turn['role']}] {turn['content']}")
print("\n" + "─" * 60 + "\n")

README.md:   0%|          | 0.00/482 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/109M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/44.4M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.02M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/573689 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6375 [00:00<?, ? examples/s]

[system] You are CiSiMi created by Lyte from KandirResearch, you respond in text+audio interweaved!
[user] You are a person who loves ice cream and you adore dogs. How excited are you that there is a new ice cream truck in town that caters to dogs.
[assistant] I am absolutely over the moon! A new ice cream truck in town is always a reason to celebrate, but one that caters to dogs? That's just the icing on the cake - or should I say, the sprinkles on the ice cream cone! I have a furry best friend at home, a sweet little pup named Max, and I know he's going to go wild for this. I can already imagine the look on his face when he tries his first dog-friendly ice cream. I'm planning on being the first in line when the truck opens, with Max by my side, of course. I've heard they have flavors like peanut butter and banana, and even a special "pup-cup" just for our canine friends. This is going to be the best summer ever!
[user] Who is Max?
[assistant] Max is my lovable and adorable golden ret

In [ ]:
# max_new_tokens=1024,do_sample=False,repetition_penalty=1.05,
# ── shared translation prompt builder ────────────────────────────────────────
def build_translation_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "Translate the following English text into Moroccan Darija "
                "(الدارجة المغربية). Preserve the meaning faithfully. "
                "Output ONLY the Darija translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ── approach A: translate full turn content as-is ────────────────────────────
def translate_full(text, max_new_tokens=1024):
    prompt = build_translation_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return decoded


# ── approach B: split on newlines, translate each chunk, rejoin ─────────────
def translate_by_lines(text):
    lines = text.split("\n")
    translated = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            translated.append("")
            continue
        translated.append(translate_full(stripped))

    return "\n".join(translated)


# ── approach C: split on sentences, translate each, rejoin ──────────────────
def split_sentences(text):
    # split on sentence boundaries while keeping punctuation attached
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def translate_by_sentences(text):
    sentences = split_sentences(text)
    if not sentences:
        return ""
    return " ".join(translate_full(s) for s in sentences)


# ── smart strategy ───────────────────────────────────────────────────────────
def smart_translate(text):
    stripped = text.strip()

    # keep full context for short/single-block text
    non_empty_lines = [l for l in stripped.split("\n") if l.strip()]

    if len(non_empty_lines) > 1:
        return translate_by_lines(stripped)

    return translate_full(stripped)


# ── test on first 3 samples ──────────────────────────────────────────────────
for i, example in enumerate(das.select(range(3))):
    print(f"═══ Example {i + 1} ═══")

    for turn in example["messages"]:
        if turn["role"] == "system":
            continue

        original = turn["content"]
        token_count = len(tokenizer.encode(original, add_special_tokens=False))

        print(f"\n  [{turn['role']}] ({token_count} tokens)")
        print(f"ORIGINAL        : {original}")
        print(f"SMART TRANSLATE : {smart_translate(original)}")

    print()

═══ Example 1 ═══

  [user] (33 tokens)
ORIGINAL        : You are a person who loves ice cream and you adore dogs. How excited are you that there is a new ice cream truck in town that caters to dogs.
SMART TRANSLATE : نتا واحد كيبغي الآيس كريم وكيعشق الكلاب. شحال فرحان نتا بلي كاين طوموبيل جديدة ديال الآيس كريم فالمدينة اللي كتصاوب حتى للكلاب.

  [assistant] (161 tokens)
ORIGINAL        : I am absolutely over the moon! A new ice cream truck in town is always a reason to celebrate, but one that caters to dogs? That's just the icing on the cake - or should I say, the sprinkles on the ice cream cone! I have a furry best friend at home, a sweet little pup named Max, and I know he's going to go wild for this. I can already imagine the look on his face when he tries his first dog-friendly ice cream. I'm planning on being the first in line when the truck opens, with Max by my side, of course. I've heard they have flavors like peanut butter and banana, and even a special "pup-cup" just for our

In [ ]:
#max_new_tokens=1024,do_sample=True,temperature=0.2,top_p=0.9,repetition_penalty=1.05,
# ── shared translation prompt builder ────────────────────────────────────────
def build_translation_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "Translate the following English text into Moroccan Darija "
                "(الدارجة المغربية). Preserve the meaning faithfully. "
                "Output ONLY the Darija translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ── approach A: translate full turn content as-is ────────────────────────────
def translate_full(text, max_new_tokens=1024):
    prompt = build_translation_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return decoded


# ── approach B: split on newlines, translate each chunk, rejoin ─────────────
def translate_by_lines(text):
    lines = text.split("\n")
    translated = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            translated.append("")
            continue
        translated.append(translate_full(stripped))

    return "\n".join(translated)


# ── approach C: split on sentences, translate each, rejoin ──────────────────
def split_sentences(text):
    # split on sentence boundaries while keeping punctuation attached
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def translate_by_sentences(text):
    sentences = split_sentences(text)
    if not sentences:
        return ""
    return " ".join(translate_full(s) for s in sentences)


# ── smart strategy ───────────────────────────────────────────────────────────
def smart_translate(text):
    stripped = text.strip()

    # keep full context for short/single-block text
    non_empty_lines = [l for l in stripped.split("\n") if l.strip()]

    if len(non_empty_lines) > 1:
        return translate_by_lines(stripped)

    return translate_full(stripped)


# ── test on first 3 samples ──────────────────────────────────────────────────
for i, example in enumerate(das.select(range(3))):
    print(f"═══ Example {i + 1} ═══")

    for turn in example["messages"]:
        if turn["role"] == "system":
            continue

        original = turn["content"]
        token_count = len(tokenizer.encode(original, add_special_tokens=False))

        print(f"\n  [{turn['role']}] ({token_count} tokens)")
        print(f"ORIGINAL        : {original}")
        print(f"SMART TRANSLATE : {smart_translate(original)}")

    print()

═══ Example 1 ═══

  [user] (33 tokens)
ORIGINAL        : You are a person who loves ice cream and you adore dogs. How excited are you that there is a new ice cream truck in town that caters to dogs.
SMART TRANSLATE : نتا واحد بنادم كيبغي الآيس كريم وكيعشق الكلاب. شحال فرحان نتا بلي كاين طوموبيل جديدة ديال الآيس كريم فالمدينة اللي كتصاوب حتى للكلاب.

  [assistant] (161 tokens)
ORIGINAL        : I am absolutely over the moon! A new ice cream truck in town is always a reason to celebrate, but one that caters to dogs? That's just the icing on the cake - or should I say, the sprinkles on the ice cream cone! I have a furry best friend at home, a sweet little pup named Max, and I know he's going to go wild for this. I can already imagine the look on his face when he tries his first dog-friendly ice cream. I'm planning on being the first in line when the truck opens, with Max by my side, of course. I've heard they have flavors like peanut butter and banana, and even a special "pup-cup" just f

In [ ]:
#max_new_tokens=1024,do_sample=True,temperature=0.3,top_p=0.98,repetition_penalty=1.05,top_k=300,min_p=0.05,
# ── shared translation prompt builder ────────────────────────────────────────
def build_translation_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "Translate the following English text into Moroccan Darija "
                "(الدارجة المغربية). Preserve the meaning faithfully. "
                "Output ONLY the Darija translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# ── approach A: translate full turn content as-is ────────────────────────────
def translate_full(text, max_new_tokens=1024):
    prompt = build_translation_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.98,
            repetition_penalty=1.05,
            top_k=300,
            min_p=0.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return decoded


# ── approach B: split on newlines, translate each chunk, rejoin ─────────────
def translate_by_lines(text):
    lines = text.split("\n")
    translated = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            translated.append("")
            continue
        translated.append(translate_full(stripped))

    return "\n".join(translated)


# ── approach C: split on sentences, translate each, rejoin ──────────────────
def split_sentences(text):
    # split on sentence boundaries while keeping punctuation attached
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def translate_by_sentences(text):
    sentences = split_sentences(text)
    if not sentences:
        return ""
    return " ".join(translate_full(s) for s in sentences)


# ── smart strategy ───────────────────────────────────────────────────────────
def smart_translate(text):
    stripped = text.strip()

    # keep full context for short/single-block text
    non_empty_lines = [l for l in stripped.split("\n") if l.strip()]

    if len(non_empty_lines) > 1:
        return translate_by_lines(stripped)

    return translate_full(stripped)


# ── test on first 3 samples ──────────────────────────────────────────────────
for i, example in enumerate(das.select(range(3))):
    print(f"═══ Example {i + 1} ═══")

    for turn in example["messages"]:
        if turn["role"] == "system":
            continue

        original = turn["content"]
        token_count = len(tokenizer.encode(original, add_special_tokens=False))

        print(f"\n  [{turn['role']}] ({token_count} tokens)")
        print(f"ORIGINAL        : {original}")
        print(f"SMART TRANSLATE : {smart_translate(original)}")

    print()

═══ Example 1 ═══

  [user] (33 tokens)
ORIGINAL        : You are a person who loves ice cream and you adore dogs. How excited are you that there is a new ice cream truck in town that caters to dogs.
SMART TRANSLATE : نتا واحد الشخص كيبغي الآيس كريم وكيعشق الكلاب. شحال فرحان نتا بلي كاين طوموبيل جديدة ديال الآيس كريم فالحومة اللي كتصاوب حتى للكلاب.

  [assistant] (161 tokens)
ORIGINAL        : I am absolutely over the moon! A new ice cream truck in town is always a reason to celebrate, but one that caters to dogs? That's just the icing on the cake - or should I say, the sprinkles on the ice cream cone! I have a furry best friend at home, a sweet little pup named Max, and I know he's going to go wild for this. I can already imagine the look on his face when he tries his first dog-friendly ice cream. I'm planning on being the first in line when the truck opens, with Max by my side, of course. I've heard they have flavors like peanut butter and banana, and even a special "pup-cup" just fo

In [ ]:
#max_new_tokens=512,do_sample=True,temperature=0.3,top_p=0.95,repetition_penalty=1.1,
# ══════════════════════════════════════════════════════════════════════
# FULLY UNSEEN TEST CASES — none of these appear in MANUAL_FIXES
# ══════════════════════════════════════════════════════════════════════

tests_en_to_dr = [
    # ── Strict instruction & formatting ──────────────────────────────
    "Reply with only the word 'Accepted'. Do not add punctuation or explanation.",
    "Write exactly 3 sentences about rain. Each sentence must start with 'The'.",
    "Give me a numbered list of 4 Moroccan cities. Nothing else.",
    "End your response with the exact string: [COMPLETE]. No words after it.",
    "Do not translate the word 'server'. Keep it in English inside the sentence.",

    # ── Idioms (paraphrased / unseen variants) ────────────────────────
    "Stop beating around the bush and tell me what happened.",
    "She was walking on eggshells around her boss all week.",
    "He's got a chip on his shoulder about not getting promoted.",
    "We need to get the ball rolling on this project before Friday.",
    "That exam was a walk in the park compared to the last one.",
    "You can't judge a book by its cover.",
    "He burned his bridges when he quit without notice.",
    "She's sitting on the fence about whether to accept the offer.",

    # ── 3rd person narrative & gender tracking ────────────────────────
    "Layla forgot her umbrella at the office. When she realized, it was already raining. Her colleague Rachid offered to drive her home.",
    "Omar asked his wife to pick up the kids from school. She called him back and said she was stuck in a meeting.",
    "The nurse checked the patient's blood pressure. She told him everything looked normal.",
    "Hassan and his brother Karim opened a bakery together. Their mother was their first customer.",

    # ── Entity / value preservation ───────────────────────────────────
    "The concert tickets cost $85.00 each and the venue is Madison Square Garden in New York.",
    "Send the report to john.smith@company.org before 5:00 PM on April 3rd, 2025.",
    "The Samsung Galaxy S24 Ultra has a 200MP camera and costs $1,299.",
    "Dr. Fatima Zahra received the Nobel Prize in Chemistry in Stockholm, Sweden.",
    "The password for the admin panel is Tr0ub4dor&3. Do not share it.",

    # ── Technical / code preservation ─────────────────────────────────
    "The log shows: FATAL: role 'postgres' does not exist.",
    "Add `export PATH=$HOME/.local/bin:$PATH` to your `.bashrc` file.",
    "The regex pattern `^[a-zA-Z0-9._%+-]+@` matches the local part of an email.",
    "Debug the function with `console.log(JSON.stringify(data, null, 2))`.",
    "The Docker container exited with code 137 (OOMKilled).",

    # ── Negation / subtle meaning ─────────────────────────────────────
    "I didn't say I wouldn't help, I said I couldn't help today.",
    "She's not unqualified, she just lacks experience in that specific area.",
    "It's not impossible, it's just very unlikely.",
    "He never claimed to be an expert, people just assumed he was.",

    # ── Daily life / Moroccan context ─────────────────────────────────
    "The pharmacy is closed, I need to find another one to buy cough syrup.",
    "My neighbor's rooster wakes me up every morning at 5 AM.",
    "The couscous on Friday at my aunt's house is the best tradition ever.",
    "I left my wallet in the petit taxi and the driver returned it.",
    "The electricity went out again during the storm last night.",

    # ── Quoted speech ─────────────────────────────────────────────────
    'She said, "I will be there at exactly 8:00 PM, don\'t be late."',
    'The doctor told him, "You need to drink more water and sleep at least 7 hours."',
    'My mom always says, "Early to bed, early to rise."',

    # ── Numbers / units / currencies ──────────────────────────────────
    "The apartment is 120 m² and rents for 6,500 MAD per month.",
    "Mix 250g of flour with 100ml of milk and bake at 180°C for 30 minutes.",
    "The flight from Casablanca to Paris takes 3 hours and 15 minutes. The ticket costs €350.",

    # ── Multi-sentence coherent paragraph ─────────────────────────────
    "Last summer, we drove from Marrakech to Essaouira along the coast. The road was beautiful but narrow in some places. We stopped twice for tea and arrived just before sunset. The kids ran straight to the beach.",
    "Artificial intelligence is changing how we work. Many jobs that existed ten years ago are now automated. However, new opportunities are also being created. The key is to keep learning and adapting.",
]

tests_dr_to_en = [
    # ── Everyday conversation ─────────────────────────────────────────
    "مشيت للحانوت ولقيتو مسدود. درت دورة كبيرة باش نلقى شي حانوت آخر.",
    "ولدي مابغاش ياكل الخضرة. كل يوم نفس الشي.",
    "الجيران ديالنا دارو عرس كبير البارح. الموسيقى بقات حتى لنص الليل.",
    "خويا سيفط ليا رسالة وقال ليا بلي غادي يجي يزورنا هاد الويكاند.",

    # ── Technical ──────────────────────────────────────────────────────
    "البروڭرام وقف وعطاني هاد الخطأ: MemoryError: Unable to allocate array.",
    "خاصك تدير `docker-compose up -d` باش تخدم السيرفر فالخلفية.",

    # ── Idioms & expressions ──────────────────────────────────────────
    "ضرب الحديد وهو سخون قبل ما تطيح الفرصة.",
    "هاد المشكل كبر بزاف، ما بقيناش عارفين من فين نبداو.",
    "كيلعب بالعافية وما كيسمعش لحتى واحد.",

    # ── Values / entities ─────────────────────────────────────────────
    "الرحلة من فاس لطنجة كتاخد تقريبا 3 ديال السوايع بالطريق السيار. الثمن ديال البيياج هو 120 درهم.",
    "شريت واحد الهاتف Samsung Galaxy A54 من جوميا ب 2,800 درهم.",

    # ── Narrative / gender ────────────────────────────────────────────
    "فاطمة الزهراء خدات الباك ب 18.5. والديها فرحو بيها بزاف وداروها عشا كبير.",
    "سعيد نسى الباسبور ديالو فالدار ورجع من المطار خاوي اليدين.",
]


# ── Run tests ─────────────────────────────────────────────────────────
SYS_EN_DR = (
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the Darija translation."
)

SYS_DR_EN = (
    "Translate the following Moroccan Darija (الدارجة المغربية) text into English. "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the English translation."
)


def infer(system_prompt, text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to(model.device)

    outputs = model.generate(
        input_ids,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.3,
        top_p=0.95,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen_text = tokenizer.decode(
        outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return gen_text.strip()


print("\n" + "=" * 60)
print("  UNSEEN TESTS: ENGLISH -> MOROCCAN DARIJA")
print("=" * 60)
for text in tests_en_to_dr:
    print(f"\n[EN] {text}")
    print(f"[DR] {infer(SYS_EN_DR, text)}")

print("\n" + "=" * 60)
print("  UNSEEN TESTS: MOROCCAN DARIJA -> ENGLISH")
print("=" * 60)
for text in tests_dr_to_en:
    print(f"\n[DR] {text}")
    print(f"[EN] {infer(SYS_DR_EN, text)}")



  UNSEEN TESTS: ENGLISH -> MOROCCAN DARIJA

[EN] Reply with only the word 'Accepted'. Do not add punctuation or explanation.
[DR] جاوب غير بكلمة 'Accepted'. ما تزيد حتى علامة ولا شرح.

[EN] Write exactly 3 sentences about rain. Each sentence must start with 'The'.
[DR] كتب بالضبط 3 ديال الجمل على الشتا. كل جملة خاص تبدا ب'الشتي'.

[EN] Give me a numbered list of 4 Moroccan cities. Nothing else.
[DR] عطيني ليستة مرقمة فيها 4 ديال المدن المغربية. ما تزيد حتى حاجة خرى غير المدن اللي طالبين.

[EN] End your response with the exact string: [COMPLETE]. No words after it.
[DR] سالي الجواب ديالك ببالضبط السلسلة: [COMPLETE]. ما تزيد حتى كلمة من بعدها.

[EN] Do not translate the word 'server'. Keep it in English inside the sentence.
[DR] ما تترجم كلمة 'server' للفرنسية. خليها بالإنجليزية داخل الجملة.

[EN] Stop beating around the bush and tell me what happened.
[DR] حبس الكذّاب وقول ليا شنو واقع.

[EN] She was walking on eggshells around her boss all week.
[DR] كانت كتمشى على البيضات ديال البوس 

In [ ]:
#max_new_tokens=512, temperature=temp,top_p=0.9, do_sample=True
end_response_id = tokenizer.convert_tokens_to_ids("<|END_RESPONSE|>")
end_turn_id = tokenizer.convert_tokens_to_ids("<|END_OF_TURN_TOKEN|>")
eos_id = tokenizer.eos_token_id
stop_ids = [t for t in [eos_id, end_response_id, end_turn_id] if t is not None]

def translate(text, sys_prompt, temp=0.3):
    msgs = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, temperature=temp,
            top_p=0.9, do_sample=True, eos_token_id=stop_ids,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


SYS_EN_DR = "Translate the following English text into Moroccan Darija (الدارجة المغربية). Output ONLY the Darija translation."
SYS_DR_EN = "Translate the following Moroccan Darija (الدارجة المغربية) text into English. Output ONLY the English translation."
SYS_FR_DR = "Translate the following French text into Moroccan Darija. Output ONLY the Darija translation."
SYS_DR_FR = "Translate the following Moroccan Darija text into French. Output ONLY the French translation."
SYS_AR_DR = "Translate the following Modern Standard Arabic text into Moroccan Darija. Output ONLY the Darija translation."
SYS_DR_AR = "Translate the following Moroccan Darija text into Modern Standard Arabic. Output ONLY the Arabic translation."
SYS_EN_FR = "Translate the following English text into French. Output ONLY the French translation."
SYS_FR_EN = "Translate the following French text into English. Output ONLY the English translation."


# ============================================================================
# TEST 1: Previously BROKEN EN→DR (seen during training)
# ============================================================================
print("\n" + "="*70)
print("  TEST 1: 🔧 Previously BROKEN EN→DR (SEEN in training)")
print("="*70)

seen_tests = [
    "I love Morocco.",
    "The weather is very hot today.",
    "Hello",
    "Bro the WiFi is so slow today I can't even load a YouTube video.",
    "I'm so tired, I barely slept last night and I have an exam tomorrow.",
    "Let's go grab some food, I'm starving.",
    "I failed my exam today and I feel terrible. My parents are going to be so disappointed.",
    "I'm so happy, I got accepted to the university I wanted!",
    "My grandmother makes the best tagine in the world.",
    "The Moroccan national football team made history by reaching the semi-finals of the 2022 FIFA World Cup.",
]

for text in seen_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text}")
    print(f"  🇲🇦 {r}")


# ============================================================================
# TEST 2: UNSEEN EN→DR (generalization — NOT in training data)
# ============================================================================
print("\n" + "="*70)
print("  TEST 2: 🔬 UNSEEN EN→DR (NOT in training data)")
print("="*70)

unseen_en_dr = [
    # Short
    "I love Tangier.",
    "I love Rabat.",
    "I'm bored.",
    "The internet is not working.",
    "I'm thirsty.",
    "Call me later.",
    "I'm at the airport.",
    "The food is cold.",
    "Turn off the lights.",
    "I don't like this movie.",

    # Medium
    "My mother cooks the best rfissa in the world.",
    "Can you lend me some money until the end of the month?",
    "I woke up late and missed the bus to school.",
    "My little sister won't stop crying, she wants ice cream.",
    "The electricity went out in the whole neighborhood.",
    "I'm looking for a cheap apartment in Rabat.",
    "She graduated from university last year and still hasn't found a job.",
    "We're going to Chefchaouen this weekend, want to come with us?",

    # Long
    "The new tramway line in Casablanca has made it much easier for people to commute between the city center and the suburbs.",
    "My grandfather used to tell me stories about how life was different in Morocco fifty years ago.",

    # Casual/slang
    "Dude I just saw the craziest thing at the market.",
    "This class is so boring I'm about to fall asleep.",
    "No way, you're lying! That can't be real.",

    # Technical
    "You need to restart the router and wait 30 seconds before reconnecting.",
    "The new iPhone costs 15,000 dirhams, that's way too expensive.",
]

for text in unseen_en_dr:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text}")
    print(f"  🇲🇦 {r}")


# ============================================================================
# TEST 3: DR→EN (was always good, check it didn't break)
# ============================================================================
print("\n" + "="*70)
print("  TEST 3: ✅ DR→EN (regression check)")
print("="*70)

dr_en_tests = [
    "كنبغي المغرب.",
    "الجو سخون بزاف اليوم.",
    "واش النعاس ديال النهار كيعوض النعاس ديال الليل؟",
    "أنا فرحان بزاف حيت نجحت فالامتحان.",
    "المغرب عندو بزاف ديال الثروات المعدنية، خصوصاً الفوسفاط.",
    "خاصك تشرب بزاف ديال الما فالصيف حيت الجسم كيخسر الما فاش كتعرق.",
    "كيفاش نقدر نتعلم البرمجة بوحدي؟",
    "الذكاء الاصطناعي غادي يبدل بزاف ديال الحوايج فحياتنا.",
    # Unseen DR→EN
    "نسيت الباسبور ديالي فالدار.",
    "الكوزينة ديال ماما هي أحسن حاجة فالعالم.",
    "مشيت للسبيطار حيت كان عندي وجع كبير فالكرش.",
]

for text in dr_en_tests:
    r = translate(text, SYS_DR_EN)
    print(f"\n  🇲🇦 {text}")
    print(f"  🇬🇧 {r}")


# ============================================================================
# TEST 4: FR↔DR (was completely broken)
# ============================================================================
print("\n" + "="*70)
print("  TEST 4: 🇫🇷↔🇲🇦 French ↔ Darija")
print("="*70)

# FR→DR seen
fr_dr_seen = [
    "Le temps au Maroc est très chaud pendant les mois d'été.",
    "J'aime le Maroc.",
    "Merci beaucoup.",
    "Je suis fatigué.",
]

# FR→DR unseen
fr_dr_unseen = [
    "Je suis en retard pour le travail.",
    "Ma voiture est tombée en panne sur l'autoroute.",
    "Les prix ont beaucoup augmenté cette année.",
    "Mon fils a réussi son baccalauréat avec mention.",
    "Je cherche un bon restaurant de poisson à Essaouira.",
]

print("\n  --- FR→DR (seen) ---")
for text in fr_dr_seen:
    r = translate(text, SYS_FR_DR)
    print(f"\n  🇫🇷 {text}")
    print(f"  🇲🇦 {r}")

print("\n  --- FR→DR (UNSEEN) ---")
for text in fr_dr_unseen:
    r = translate(text, SYS_FR_DR)
    print(f"\n  🇫🇷 {text}")
    print(f"  🇲🇦 {r}")

# DR→FR
dr_fr_tests = [
    "كنبغي المغرب.",
    "الجو سخون بزاف اليوم.",
    "أنا عيان.",
    "المغرب عندو بزاف ديال الثروات المعدنية، خصوصاً الفوسفاط.",
    # Unseen
    "نسيت الباسبور ديالي فالدار.",
    "بغيت نمشي للبحر هاد الويكاند.",
]

print("\n  --- DR→FR ---")
for text in dr_fr_tests:
    r = translate(text, SYS_DR_FR)
    print(f"\n  🇲🇦 {text}")
    print(f"  🇫🇷 {r}")


# ============================================================================
# TEST 5: AR↔DR
# ============================================================================
print("\n" + "="*70)
print("  TEST 5: 🇸🇦↔🇲🇦 Arabic MSA ↔ Darija")
print("="*70)

ar_dr_tests = [
    ("ar_dr", "الذكاء الاصطناعي يغير طريقة تعاملنا مع البحث العلمي."),
    ("ar_dr", "كيف حالك؟"),
    ("ar_dr", "أحب المغرب."),
    # Unseen
    ("ar_dr", "هل يمكنك مساعدتي في حل هذه المسألة؟"),
    ("ar_dr", "الطقس بارد جداً في الشتاء."),
    ("ar_dr", "أريد أن أتعلم اللهجة المغربية."),
]

for direction, text in ar_dr_tests:
    r = translate(text, SYS_AR_DR)
    print(f"\n  🇸🇦 {text}")
    print(f"  🇲🇦 {r}")

dr_ar_tests = [
    "لاباس عليك؟",
    "كنبغي المغرب.",
    "واش النعاس ديال النهار كيعوض النعاس ديال الليل؟",
    # Unseen
    "بغيت نتعلم الدارجة المغربية.",
    "الماكلة المغربية هي أحسن ماكلة فالعالم.",
]

for text in dr_ar_tests:
    r = translate(text, SYS_DR_AR)
    print(f"\n  🇲🇦 {text}")
    print(f"  🇸🇦 {r}")


# ============================================================================
# TEST 6: EN↔FR (retained ability?)
# ============================================================================
print("\n" + "="*70)
print("  TEST 6: 🇬🇧↔🇫🇷 English ↔ French (retention)")
print("="*70)

en_fr_tests = [
    "Artificial intelligence is transforming healthcare.",
    "The weather is nice today.",
    "I love Morocco.",
    # Unseen
    "The stock market crashed yesterday and many investors lost money.",
    "Can you recommend a good book about machine learning?",
]

for text in en_fr_tests:
    r = translate(text, SYS_EN_FR)
    print(f"\n  🇬🇧 {text}")
    print(f"  🇫🇷 {r}")

fr_en_tests = [
    "L'intelligence artificielle transforme le secteur de la santé.",
    "Le Maroc est un beau pays.",
    # Unseen
    "La bourse a chuté hier et beaucoup d'investisseurs ont perdu de l'argent.",
    "Je voudrais réserver une chambre d'hôtel pour deux nuits.",
]

for text in fr_en_tests:
    r = translate(text, SYS_FR_EN)
    print(f"\n  🇫🇷 {text}")
    print(f"  🇬🇧 {r}")


# ============================================================================
# TEST 7: Round-trip EN→DR→EN
# ============================================================================
print("\n" + "="*70)
print("  TEST 7: 🔄 Round-trip EN→DR→EN (all unseen)")
print("="*70)

roundtrip = [
    "My phone fell in the water and now it won't turn on.",
    "The best thing about Ramadan is the family gatherings at iftar.",
    "I want to learn how to make pastilla from scratch.",
    "The taxi driver took the long route and charged me double.",
    "She speaks four languages fluently: Arabic, French, English, and Spanish.",
]

for text in roundtrip:
    dr = translate(text, SYS_EN_DR)
    back = translate(dr, SYS_DR_EN)
    print(f"\n  🇬🇧 Original: {text}")
    print(f"  🇲🇦 Darija:   {dr}")
    print(f"  🇬🇧 Back:     {back}")


# ============================================================================
# TEST 8: System prompt variations (same input, different prompts)
# ============================================================================
print("\n" + "="*70)
print("  TEST 8: 🔀 System Prompt Variations")
print("="*70)

test_input = "I forgot my wallet at the restaurant."

prompts = [
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). Output ONLY the Darija translation.",
    "ترجم النص الإنجليزي التالي إلى الدارجة المغربية. أعطني غير الترجمة.",
    "Translate to Darija:",
    "Convert this English text to Moroccan Darija. Only output the translation.",
    "English → Darija. Translate:",
]

for sp in prompts:
    r = translate(test_input, sp)
    print(f"\n  📋 {sp[:50]}...")
    print(f"  🇬🇧 {test_input}")
    print(f"  🇲🇦 {r}")


# ============================================================================
# TEST 9: Edge cases
# ============================================================================
print("\n" + "="*70)
print("  TEST 9: ⚡ Edge Cases")
print("="*70)

edge = {
    "Single word": "Goodbye",
    "Numbers": "I have 7 brothers and the oldest is 35 years old.",
    "Mixed lang": "I need to update my CV on LinkedIn ASAP.",
    "Medical": "I have high blood pressure and the doctor prescribed me medication.",
    "Legal": "You need to sign the contract before the deadline on March 15th.",
    "Proverb-like": "Not everything that glitters is gold.",
    "Question": "Why is the sky blue?",
    "Command": "Stop talking and listen to me.",
    "Sarcasm": "Oh great, another meeting that could have been an email.",
    "Very short": "No thanks.",
    "Greeting chain": "Hi, how are you, long time no see!",
}

for label, text in edge.items():
    r = translate(text, SYS_EN_DR)
    print(f"\n  [{label}]")
    print(f"  🇬🇧 {text}")
    print(f"  🇲🇦 {r}")


print(f"\n{'='*70}")
print("  ✅ ALL TESTS COMPLETE")
print(f"{'='*70}")


  TEST 1: 🔧 Previously BROKEN EN→DR (SEEN in training)

  🇬🇧 I love Morocco.
  🇲🇦 كنبغي المغرب.

  🇬🇧 The weather is very hot today.
  🇲🇦 الجو سخون بزاف اليوم.

  🇬🇧 Hello
  🇲🇦 سلام

  🇬🇧 Bro the WiFi is so slow today I can't even load a YouTube video.
  🇲🇦 خويا الويفي بطيء بزاف اليوم وما كنقدرش حتى نحمل فيديو ديال يوتيوب.

  🇬🇧 I'm so tired, I barely slept last night and I have an exam tomorrow.
  🇲🇦 أنا عيان بزاف، بالليل ما نعستش والامتحان غدا.

  🇬🇧 Let's go grab some food, I'm starving.
  🇲🇦 يلاه نمشيو ناكلو شي حاجة، فيا الجوع بزاف.

  🇬🇧 I failed my exam today and I feel terrible. My parents are going to be so disappointed.
  🇲🇦 سقطت فالامتحان اليوم وحاسس براسي خايب بزاف. الوالدين ديالي غادي يكونو مخيبانين فيا.

  🇬🇧 I'm so happy, I got accepted to the university I wanted!
  🇲🇦 أنا فرحان بزاف، تقبلت فالجامعة اللي كنت باغيها!

  🇬🇧 My grandmother makes the best tagine in the world.
  🇲🇦 جداتي كتصاوب أحسن طاجين فالعالم.

  🇬🇧 The Moroccan national football team made history by reac

In [ ]:
#max_new_tokens=1024, temperature=0.3, top_p=0.9, do_sample=True
def translate(text, sys_prompt, temp=0.3):
    msgs = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1024, temperature=temp, top_p=0.9, do_sample=True, eos_token_id=stop_ids)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# ============================================================================
# TEST 1: MATH & REASONING
# ============================================================================
print("\n" + "="*70)
print("  TEST 1: 🔢 Math & Reasoning")
print("="*70)

math_tests = [
    # Step-by-step reasoning
    "If a train travels at 120 km/h and needs to cover 360 km, how long will the journey take? Let's think step by step: First, we use the formula time = distance / speed. So time = 360 / 120 = 3 hours.",

    # Algebra
    "To solve the equation 2x + 5 = 17, first subtract 5 from both sides to get 2x = 12, then divide both sides by 2 to get x = 6.",

    # Word problem
    "Ahmed has 3 boxes. Each box contains 12 oranges. He gives 7 oranges to his neighbor and eats 2 himself. How many oranges does he have left? Total oranges: 3 × 12 = 36. Oranges given away and eaten: 7 + 2 = 9. Remaining: 36 - 9 = 27 oranges.",

    # Fractions
    "To add the fractions 1/3 and 1/4, we need a common denominator. The least common multiple of 3 and 4 is 12. So 1/3 = 4/12 and 1/4 = 3/12. Therefore 1/3 + 1/4 = 4/12 + 3/12 = 7/12.",

    # Probability
    "If you roll two dice, the probability of getting a sum of 7 is 6/36 = 1/6, because there are 6 favorable outcomes: (1,6), (2,5), (3,4), (4,3), (5,2), (6,1).",

    # Geometry
    "The area of a circle with radius 5 cm is calculated using the formula A = πr². So A = π × 5² = 25π ≈ 78.54 square centimeters.",

    # Logic puzzle
    "If all cats are animals, and some animals are pets, can we conclude that some cats are pets? No, this is a logical fallacy. The premises don't guarantee that any cats are in the subset of animals that are pets.",
]

for text in math_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:150]}...")


# ============================================================================
# TEST 2: CODE & PROGRAMMING
# ============================================================================
print("\n" + "="*70)
print("  TEST 2: 💻 Code & Programming")
print("="*70)

code_tests = [
    # Python explanation
    "In Python, a list comprehension like [x**2 for x in range(10)] creates a new list containing the squares of numbers from 0 to 9.",

    # Debugging
    "The error 'IndexError: list index out of range' means you're trying to access an element at a position that doesn't exist in the list. For example, if your list has 5 elements, valid indices are 0 through 4.",

    # Algorithm
    "Binary search works by repeatedly dividing the search interval in half. Start with the middle element. If the target is less than the middle element, search the left half. If greater, search the right half. This gives O(log n) time complexity.",

    # SQL
    "To find all users who registered in 2024, you can use: SELECT * FROM users WHERE YEAR(created_at) = 2024 ORDER BY created_at DESC.",

    # Git
    "To undo the last commit while keeping your changes, use 'git reset --soft HEAD~1'. This moves the HEAD pointer back one commit but keeps all changes staged.",

    # API concept
    "A REST API uses HTTP methods to perform operations: GET to retrieve data, POST to create new data, PUT to update existing data, and DELETE to remove data.",

    # Data structures
    "A hash table stores key-value pairs and uses a hash function to compute an index. The average time complexity for insertion, deletion, and lookup is O(1), but worst case is O(n) due to collisions.",

    # Code with inline explanation
    "The function def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2) calculates the nth Fibonacci number recursively, but it has exponential time complexity O(2^n) because it recalculates the same values many times.",
]

for text in code_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:150]}...")


# ============================================================================
# TEST 3: SCIENCE
# ============================================================================
print("\n" + "="*70)
print("  TEST 3: 🔬 Science")
print("="*70)

science_tests = [
    # Physics
    "Newton's second law states that force equals mass times acceleration (F = ma). If a 10 kg object accelerates at 3 m/s², the net force acting on it is 30 Newtons.",

    # Chemistry
    "Photosynthesis is the process by which plants convert carbon dioxide and water into glucose and oxygen using sunlight. The chemical equation is: 6CO₂ + 6H₂O + light energy → C₆H₁₂O₆ + 6O₂.",

    # Biology
    "DNA replication is semi-conservative, meaning each new double helix contains one original strand and one newly synthesized strand. The enzyme helicase unwinds the double helix, and DNA polymerase adds complementary nucleotides.",

    # Medicine
    "Type 2 diabetes occurs when the body becomes resistant to insulin or doesn't produce enough insulin. Risk factors include obesity, sedentary lifestyle, and family history. Treatment typically involves lifestyle changes, oral medications like metformin, and sometimes insulin injections.",

    # Astronomy
    "A black hole forms when a massive star collapses under its own gravity at the end of its life. The gravitational pull is so strong that nothing, not even light, can escape once it crosses the event horizon.",

    # Environmental science
    "The greenhouse effect works like this: solar radiation passes through the atmosphere and warms Earth's surface. The surface emits infrared radiation, which is absorbed by greenhouse gases like CO₂ and methane, trapping heat in the atmosphere.",

    # Neuroscience
    "Neurons communicate through synapses. When an electrical signal reaches the end of a neuron, it triggers the release of neurotransmitters into the synaptic cleft. These chemicals bind to receptors on the next neuron, either exciting or inhibiting it.",
]

for text in science_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:150]}...")


# ============================================================================
# TEST 4: CHAIN-OF-THOUGHT REASONING
# ============================================================================
print("\n" + "="*70)
print("  TEST 4: 🧠 Chain-of-Thought Reasoning")
print("="*70)

cot_tests = [
    "Let's think about this step by step. If I buy 3 shirts at 150 dirhams each and 2 pants at 200 dirhams each, the total cost is: Shirts: 3 × 150 = 450 dirhams. Pants: 2 × 200 = 400 dirhams. Total: 450 + 400 = 850 dirhams. If I have a 10% discount, I save 85 dirhams, so I pay 765 dirhams.",

    "Question: Is it better to invest 10,000 dirhams at 5% simple interest for 3 years, or at 4% compound interest for 3 years? Simple interest: 10000 × 0.05 × 3 = 1500 dirhams profit. Total: 11,500. Compound interest: 10000 × (1.04)³ = 10000 × 1.124864 = 11,248.64 dirhams. Therefore, simple interest at 5% is better in this case.",

    "To determine if a number is prime, we check if it's divisible by any integer from 2 to its square root. For 97: √97 ≈ 9.85. We check divisibility by 2, 3, 5, 7. 97/2 = 48.5, 97/3 = 32.33, 97/5 = 19.4, 97/7 = 13.86. None divide evenly, so 97 is prime.",

    "The trolley problem: A runaway trolley is heading toward 5 people. You can pull a lever to divert it to a side track where it will kill 1 person. From a utilitarian perspective, pulling the lever saves the most lives (5 vs 1). But from a deontological perspective, actively causing someone's death by pulling the lever is morally wrong, even to save others.",
]

for text in cot_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:200]}...")


# ============================================================================
# TEST 5: TECHNICAL INSTRUCTIONS
# ============================================================================
print("\n" + "="*70)
print("  TEST 5: 📋 Technical Instructions")
print("="*70)

instruction_tests = [
    "Step 1: Install Python from python.org. Step 2: Open terminal and type 'pip install numpy pandas matplotlib'. Step 3: Create a new file called analysis.py. Step 4: Import the libraries with 'import numpy as np'. Step 5: Run the script with 'python analysis.py'.",

    "To set up a WiFi router: First, connect the router to your modem using an Ethernet cable. Then, plug in the power adapter and wait for the lights to stabilize. Next, connect to the default network shown on the router's label. Finally, open a browser and go to 192.168.1.1 to configure your settings.",

    "Warning: Before replacing a car battery, always disconnect the negative terminal first (the black cable) to prevent short circuits. Then disconnect the positive terminal (red cable). Install the new battery in reverse order: connect positive first, then negative.",

    "Recipe: Traditional Moroccan harira. Ingredients: 200g lentils, 200g chickpeas, 500g tomatoes, 100g flour, celery, onion, salt, pepper, turmeric, ginger, cinnamon. Instructions: Soak chickpeas overnight. Blend tomatoes. Cook lentils and chickpeas with spices for 1 hour. Add the tedouira (flour mixture) at the end and stir continuously for 10 minutes.",
]

for text in instruction_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:200]}...")


# ============================================================================
# TEST 6: COMPLEX MULTI-SENTENCE PARAGRAPHS
# ============================================================================
print("\n" + "="*70)
print("  TEST 6: 📝 Complex Paragraphs")
print("="*70)

paragraph_tests = [
    "Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. There are three main types: supervised learning, where the model learns from labeled data; unsupervised learning, where the model finds patterns in unlabeled data; and reinforcement learning, where an agent learns by interacting with an environment and receiving rewards or penalties.",

    "The Moroccan education system faces several challenges. First, there is a significant gap between urban and rural schools in terms of resources and teacher quality. Second, the language policy is complex, with Arabic used in primary school, French dominating in higher education and the job market, and Amazigh gaining official recognition. Third, unemployment among university graduates remains high, suggesting a mismatch between what is taught and what the job market demands.",

    "Blockchain is a decentralized digital ledger that records transactions across a network of computers. Each block contains a timestamp, transaction data, and a cryptographic hash of the previous block, creating an immutable chain. This technology is the foundation of cryptocurrencies like Bitcoin, but it also has applications in supply chain management, voting systems, and digital identity verification.",
]

for text in paragraph_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:250]}...")


# ============================================================================
# TEST 7: SPECIAL CHARACTERS, FORMULAS, MIXED CONTENT
# ============================================================================
print("\n" + "="*70)
print("  TEST 7: 🔣 Special Characters & Mixed Content")
print("="*70)

special_tests = [
    # Math formulas inline
    "The quadratic formula x = (-b ± √(b²-4ac)) / 2a gives us the roots of any quadratic equation ax² + bx + c = 0.",

    # Code inline
    "Use the command 'for i in range(len(array)): print(array[i])' to iterate through all elements in a Python array.",

    # URLs and tech terms
    "You can download the dataset from https://huggingface.co/datasets and use the load_dataset() function from the datasets library to load it into memory.",

    # Numbers heavy
    "In 2023, Morocco's GDP was approximately $141.1 billion, with a population of 37.8 million people. The GDP per capita was about $3,732, and the inflation rate was 6.1%.",

    # Emoji context
    "The student scored 18/20 on the math exam ✅ but only 8/20 on the physics exam ❌. The passing grade is 10/20.",

    # Mixed Arabic/English technical
    "The API endpoint /api/v1/translate accepts POST requests with a JSON body containing 'text' and 'target_language' fields. The response includes the translated text and a confidence score between 0 and 1.",
]

for text in special_tests:
    r = translate(text, SYS_EN_DR)
    print(f"\n  🇬🇧 {text[:100]}...")
    print(f"  🇲🇦 {r[:150]}...")


# ============================================================================
# TEST 8: ROUND-TRIP COMPLEX (EN→DR→EN)
# ============================================================================
print("\n" + "="*70)
print("  TEST 8: 🔄 Round-trip Complex Content")
print("="*70)

roundtrip_complex = [
    "Binary search has a time complexity of O(log n), which makes it much faster than linear search O(n) for large sorted arrays.",

    "The derivative of f(x) = 3x² + 2x - 5 is f'(x) = 6x + 2. At x = 3, the slope of the tangent line is f'(3) = 20.",

    "Photosynthesis converts CO₂ and H₂O into glucose and oxygen. Without this process, life on Earth would not be possible.",

    "To configure a firewall rule, use: iptables -A INPUT -p tcp --dport 443 -j ACCEPT. This allows incoming HTTPS traffic on port 443.",

    "The Pythagorean theorem states that in a right triangle, a² + b² = c², where c is the hypotenuse.",
]

for text in roundtrip_complex:
    dr = translate(text, SYS_EN_DR)
    back = translate(dr, SYS_DR_EN)
    print(f"\n  🇬🇧 Original: {text[:80]}...")
    print(f"  🇲🇦 Darija:   {dr[:120]}...")
    print(f"  🇬🇧 Back:     {back[:120]}...")


print(f"\n{'='*70}")
print("  ✅ ALL COMPLEX TESTS COMPLETE")
print(f"{'='*70}")


  TEST 1: 🔢 Math & Reasoning

  🇬🇧 If a train travels at 120 km/h and needs to cover 360 km, how long will the journey take? Let's thin...
  🇲🇦 إلا القطار كيمشي بـ 120 كلم فالساعة وخاصو يقطع 360 كلم، شحال ديال الوقت غادي ياخد؟ يلاه نفكرو خطوة بخطوة: أولا، كنستعملو الصيغة time = distance / spee...

  🇬🇧 To solve the equation 2x + 5 = 17, first subtract 5 from both sides to get 2x = 12, then divide both...
  🇲🇦 باش نحلو المعادلة 2x + 5 = 17، أولا نطرحو 5 من الجهتين باش نلقاو 2x = 12، من بعد نقسمو على 2 باش نلقاو x = 6....

  🇬🇧 Ahmed has 3 boxes. Each box contains 12 oranges. He gives 7 oranges to his neighbor and eats 2 himse...
  🇲🇦 أحمد عندو 3 ديال الصناديق. كل صندوق فيه 12 ديال الليمون. عطى 7 للجار ديالو وكلى 2. شحال بقاو عندو؟ المجموع: 3 × 12 = 36. اللي عطا واللي كلا: 7 + 2 = 9...

  🇬🇧 To add the fractions 1/3 and 1/4, we need a common denominator. The least common multiple of 3 and 4...
  🇲🇦 باش نجمعو الكسور 1/3 و 1/4، خاصنا مقام مشترك. أصغر مضاعف مشترك ديال 3 و 4 هو 12. إذن 1/3 